In [1]:

""" 
multitask_pipeline_pca_full.py
------------------------------
Full, resume-friendly multi-output regression pipeline with PCA compression of Y.

Major steps (each exposed as a top-level function you call separately):
  1) run_hparam_search_pca(...)  -> Minimal-refit selection of (alpha_fraction, l1_ratio)
  2) run_final_lasso_pca(...)    -> Final MT Elastic Net on PCA-scores, de-biasing + BH-FDR,
                                    validation diagnostics/plots, and CSV of kept/filtered predictors
  3) run_final_ols(...)          -> OLS on ALL outputs with selected predictors, automatic HC3
                                    if BP indicates heteroskedasticity; uncertainty, tests, plots

Design choices for speed:
  • PCA on Y reduces q to r ≪ q while preserving shared structure across tasks
    (# Jolliffe, 2002; # Hastie, Tibshirani & Friedman, 2009).
  • Single warm-started path for alpha fractions + EBIC/ALO scoring (# Chen & Chen, 2008; # Wager, Hastie & Efron, 2013).
  • KKT margin adequacy check to avoid under-regularization (# Wainwright, 2009).
  • De-biased LASSO for inference (# Zhang & Zhang, 2014; # van de Geer et al., 2014) with
    nodewise regression (# Meinshausen & Bühlmann, 2006) — batched over PCA components.
  • BH-FDR row-wise control (# Benjamini & Hochberg, 1995) to select predictors.

Author: Dylan Hettinger; 2025-10-01
"""

from __future__ import annotations

import hashlib
import json
import logging
import os
import time
import warnings
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from joblib import dump, load
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import Lasso, MultiTaskElasticNet
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import het_breuschpagan, normal_ad


def make_global_holdout_split(
    X: pd.DataFrame, Y: pd.DataFrame, holdout_frac: float = 0.10, random_state: int = 123
):
    """
    Returns (X_train, Y_train, X_holdout, Y_holdout, train_index, holdout_index).
    The holdout subset must NOT be used for hyperparameter tuning, feature selection,
    or model fitting. It is reserved strictly for final evaluation/diagnostics.
    """
    n_total = len(X)
    rs = np.random.RandomState(random_state)
    te_size = max(1, int(round(float(holdout_frac) * n_total)))
    te_pos = rs.choice(np.arange(n_total), size=te_size, replace=False)
    mask = np.ones(n_total, dtype=bool); mask[te_pos] = False
    tr_pos = np.where(mask)[0]
    X_tr, X_te = X.iloc[tr_pos], X.iloc[te_pos]
    Y_tr, Y_te = Y.iloc[tr_pos], Y.iloc[te_pos]
    return X_tr, Y_tr, X_te, Y_te, X_tr.index, X_te.index



# =====================================================================================
# Logging
# =====================================================================================

def setup_logging(level=logging.INFO):
    logging.basicConfig(
        level=level,
        format="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%H:%M:%S",
        force=True
    )

def _sec(t: float) -> str:
    return f"{t:,.2f}s"

# =====================================================================================
# Caching helpers
# =====================================================================================

def _cache_path(cache_dir: str, prefix: str, key: str) -> os.PathLike:
    os.makedirs(cache_dir, exist_ok=True)
    return os.path.join(cache_dir, f"{prefix}__{key}.joblib")

def cache_save(obj, cache_dir: str, prefix: str, key: str):
    path = _cache_path(cache_dir, prefix, key)
    dump(obj, path)

def cache_load(cache_dir: str, prefix: str, key: str):
    path = _cache_path(cache_dir, prefix, key)
    if os.path.exists(path):
        return load(path)
    return None


def dataset_signature(X: pd.DataFrame, Y: pd.DataFrame, tag: str | None) -> str:
    """
    More robust than shape/column-name fingerprints alone. This guards against
    accidentally reusing cached artifacts after changing preprocessing while the
    dataframe shapes remain unchanged.
    """
    def _frame_sig(df: pd.DataFrame) -> str:
        n = min(len(df), 50)
        head_hash = pd.util.hash_pandas_object(df.head(n), index=True).values.tobytes()
        tail_hash = pd.util.hash_pandas_object(df.tail(n), index=True).values.tobytes()
        col_hash = pd.util.hash_pandas_object(pd.Index(df.columns.astype(str))).values.tobytes()
        dtype_sig = "|".join(map(str, df.dtypes.astype(str).tolist()[:200]))
        payload = (
            str(df.shape).encode()
            + head_hash
            + tail_hash
            + col_hash
            + dtype_sig.encode()
        )
        return hashlib.sha1(payload).hexdigest()

    base = f"{_frame_sig(X)}:{_frame_sig(Y)}:{tag or 'none'}"
    return hashlib.sha1(base.encode()).hexdigest()


# =====================================================================================
# Output culling: NZV + SNR (CSV report)
# =====================================================================================

@dataclass
class OutputCullReport:
    kept_outputs: list[str]
    culled_outputs: pd.DataFrame  # columns: name, reason, std, snr
    report_csv: str

def cull_outputs_nzv_snr(
    X: pd.DataFrame, Y: pd.DataFrame,
    nzv_std_threshold: float = 1e-8,
    snr_min: float = 0.01,
    cache_dir: str = "model_cache",
    dataset_tag: str | None = None,
    resume: bool = True
) -> OutputCullReport:
    sig = dataset_signature(X, Y, dataset_tag)
    key = hashlib.sha1(json.dumps(dict(sig=sig, nzv=float(nzv_std_threshold), snr=float(snr_min))).encode()).hexdigest()
    csv_path = os.path.join(cache_dir, f"filt_outputs__{key}.csv")
    if resume and os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        kept = df[df["reason"] == "kept"]["name"].tolist()
        culled = df[df["reason"] != "kept"][["name", "reason", "std", "snr"]].copy()
        logging.info(f"[cull] loaded report: kept={len(kept)}, culled={len(culled)}")
        return OutputCullReport(kept_outputs=kept, culled_outputs=culled, report_csv=csv_path)

    # NZV
    stds = Y.std(axis=0, ddof=0)
    nzv_mask = stds <= nzv_std_threshold
    nzv_cols = stds.index[nzv_mask].tolist()

    # SNR proxy via max abs correlation with any standardized input:
    Xs = (X - X.mean(0)) / (X.std(0, ddof=0).replace(0, 1.0))
    Ys = (Y - Y.mean(0)) / (Y.std(0, ddof=0).replace(0, 1.0))
    Xs = Xs.replace([np.inf, -np.inf], 0.0).fillna(0.0)
    Ys = Ys.replace([np.inf, -np.inf], 0.0).fillna(0.0)
    C = np.abs(Xs.values.T @ Ys.values) / max(len(X) - 1, 1)  # p × q
    rmax = C.max(axis=0)
    snr = (rmax**2) / np.maximum(1 - rmax**2, 1e-12)
    snr_series = pd.Series(snr, index=Y.columns)
    snr_mask = snr_series < snr_min
    snr_cols = snr_series.index[snr_mask].tolist()

    culled = set(nzv_cols) | set(snr_cols)
    keep = [c for c in Y.columns if c not in culled]

    rows = []
    for col in Y.columns:
        rows.append(dict(
            name=col,
            reason=("NZV" if col in nzv_cols else ("low_SNR" if col in snr_cols else "kept")),
            std=float(stds.get(col, np.nan)),
            snr=float(snr_series.get(col, np.nan))
        ))
    df_report = pd.DataFrame(rows)
    os.makedirs(cache_dir, exist_ok=True)
    df_report.to_csv(csv_path, index=False)
    logging.info(f"[cull] saved report → {csv_path} | kept={len(keep)} culled={len(culled)}")
    return OutputCullReport(kept_outputs=keep, culled_outputs=df_report[df_report["reason"]!="kept"][["name","reason","std","snr"]], report_csv=csv_path)

# =====================================================================================
# Standardize/split with PCA on Y
# =====================================================================================

@dataclass
class StdPCASplit:
    Xs_tr: pd.DataFrame
    Xs_te: pd.DataFrame | None
    Ys_tr_scores: pd.DataFrame
    Ys_te_scores: pd.DataFrame | None
    x_scaler: StandardScaler
    y_scaler: StandardScaler
    pca: PCA
    tr_index: pd.Index
    te_index: pd.Index | None

def _choose_pos_rows(X: pd.DataFrame, n: int, random_state: int) -> np.ndarray:
    rs = np.random.RandomState(random_state)
    return rs.choice(np.arange(len(X)), size=min(n, len(X)), replace=False)

def standardize_split_pcaY(
    X: pd.DataFrame, Y: pd.DataFrame,
    test_size: float = 0.2,
    random_state: int = 42,
    y_pca_variance: float | None = 0.95,
    y_pca_components: int | None = None
) -> StdPCASplit:
    n = len(X)
    if test_size and test_size > 0:
        rs = np.random.RandomState(random_state)
        te_pos = rs.choice(np.arange(n), size=int(round(test_size*n)), replace=False)
        mask = np.ones(n, dtype=bool); mask[te_pos] = False
        tr_pos = np.where(mask)[0]
        X_tr, X_te = X.iloc[tr_pos], X.iloc[te_pos]
        Y_tr, Y_te = Y.iloc[tr_pos], Y.iloc[te_pos]
    else:
        tr_pos = np.arange(n); te_pos=None
        X_tr, X_te = X, None
        Y_tr, Y_te = Y, None

    xsc = StandardScaler(with_mean=True, with_std=True)
    ysc = StandardScaler(with_mean=True, with_std=True)
    Xs_tr = pd.DataFrame(xsc.fit_transform(X_tr), index=X_tr.index, columns=X_tr.columns)
    Yz_tr = pd.DataFrame(ysc.fit_transform(Y_tr), index=Y_tr.index, columns=Y_tr.columns)
    if X_te is not None:
        Xs_te = pd.DataFrame(xsc.transform(X_te), index=X_te.index, columns=X_te.columns)
        Yz_te = pd.DataFrame(ysc.transform(Y_te), index=Y_te.index, columns=Y_te.columns)
    else:
        Xs_te=None; Yz_te=None

    if y_pca_components is not None:
        pca = PCA(n_components=int(y_pca_components), svd_solver="auto", random_state=random_state)
    else:
        pca = PCA(n_components=float(y_pca_variance or 0.95), svd_solver="full", random_state=random_state)
    Ys_tr_scores = pd.DataFrame(pca.fit_transform(Yz_tr), index=Yz_tr.index)
    if Yz_te is not None:
        Ys_te_scores = pd.DataFrame(pca.transform(Yz_te), index=Yz_te.index)
    else:
        Ys_te_scores=None

    return StdPCASplit(
        Xs_tr=Xs_tr, Xs_te=Xs_te,
        Ys_tr_scores=Ys_tr_scores, Ys_te_scores=Ys_te_scores,
        x_scaler=xsc, y_scaler=ysc, pca=pca,
        tr_index=X_tr.index, te_index=None if X_te is None else X_te.index
    )

# =====================================================================================
# Alpha_max / fit / scoring on PCA scores
# =====================================================================================

def compute_alpha_max_mtl_scores(Xs_tr: pd.DataFrame, Ys_tr_scores: pd.DataFrame) -> float:
    X = Xs_tr.values; Z = Ys_tr_scores.values
    n = X.shape[0]
    S = X.T @ Z  # p × r
    row_norms = np.sqrt((S**2).sum(axis=1))
    return (row_norms.max() / n)

def fit_mtenet_once(
    Xs: pd.DataFrame, Ys_scores: pd.DataFrame,
    alpha: float, l1_ratio: float,
    max_iter: int = 3000, tol: float = 1e-2,
    warm_start_coef_: np.ndarray | None = None,
    selection: str = "random"
) -> tuple[np.ndarray, dict]:
    """
    Fit one multitask elastic-net step with dtype-consistent warm starts.

    In sklearn's coordinate-descent backend, the warm-start coefficient buffer
    must have the same dtype as X and Y. A previous version stored the warm start
    as float32 while X/Y were float64, which causes a low-level
    ``Buffer dtype mismatch`` failure on the second alpha along the path.
    """
    X_fit = np.asfortranarray(Xs.to_numpy(dtype=np.float64), dtype=np.float64)
    Y_fit = np.asfortranarray(Ys_scores.to_numpy(dtype=np.float64), dtype=np.float64)

    model = MultiTaskElasticNet(
        alpha=alpha, l1_ratio=l1_ratio, fit_intercept=False,
        max_iter=max_iter, tol=tol, selection=selection, warm_start=True
    )
    if warm_start_coef_ is not None:
        model.coef_ = np.asfortranarray(warm_start_coef_.T, dtype=X_fit.dtype)
    t0 = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        model.fit(X_fit, Y_fit)
    dt = time.perf_counter() - t0
    B = np.asarray(model.coef_.T, dtype=X_fit.dtype)  # p×r
    info = dict(n_iter=model.n_iter_, time=dt)
    return B, info

@dataclass
class PathResult:
    alphas_abs: list[float]
    alphas_frac: list[float]
    coefs: list[np.ndarray]      # p×r
    rss: list[float]
    df_rows: list[int]
    eff_L1: list[float]
    times: list[float]

def fit_path_by_fractions(
    Xs_tr: pd.DataFrame, Ys_tr_scores: pd.DataFrame,
    l1_ratio: float,
    alpha_fracs: list[float],
    max_iter: int = 3000, tol: float = 1e-2,
    selection: str = "random"
) -> PathResult:
    alpha_max = compute_alpha_max_mtl_scores(Xs_tr, Ys_tr_scores)
    alpha_fracs = sorted(alpha_fracs, reverse=True)
    alphas_abs = [ (af / max(l1_ratio, 1e-6)) * alpha_max for af in alpha_fracs ]
    coefs, rss, df_rows, times, eff = [], [], [], [], []
    warm = None
    for a_abs, a_frac in zip(alphas_abs, alpha_fracs):
        B, info = fit_mtenet_once(Xs_tr, Ys_tr_scores, alpha=a_abs, l1_ratio=l1_ratio,
                                  max_iter=max_iter, tol=tol, warm_start_coef_=warm, selection=selection)
        warm = B
        E = Ys_tr_scores.values - Xs_tr.values @ B
        rss.append(float(np.sum(E**2)))
        row_norms = np.sqrt((B**2).sum(axis=1))
        df_rows.append(int(np.count_nonzero(row_norms > 0)))
        coefs.append(B)
        times.append(info["time"])
        eff.append(a_abs * l1_ratio)
        logging.info(f"[PATH] α_frac={a_frac:.4g}, α_abs={a_abs:.6g}, df_rows={df_rows[-1]}, rss={rss[-1]:.4g}, time={_sec(times[-1])}")
    return PathResult(alphas_abs, alpha_fracs, coefs, rss, df_rows, eff, times)

def ebic_score(n: int, r: int, rss: float, df_rows: int, p: int, gamma: float = 0.5) -> float:
    k = max(df_rows, 0) * r
    sigma2 = rss / (n * r)
    return n*r*np.log(max(sigma2, 1e-15)) + k*np.log(n*r) + 2*gamma*k*np.log(max(p, 2))

def alo_rmse_active(
    Xs_tr: pd.DataFrame, Ys_tr_scores: pd.DataFrame,
    B: np.ndarray, alpha: float, l1_ratio: float
) -> float:
    X = Xs_tr.values; Z = Ys_tr_scores.values
    n, p = X.shape; r = Z.shape[1]
    row_norms = np.sqrt((B**2).sum(axis=1))
    active = np.where(row_norms > 0)[0]
    if active.size == 0:
        R = Z - X @ B
        return float(np.sqrt(np.mean(R**2)))
    XA = X[:, active]
    lam2 = alpha * max(1 - l1_ratio, 0.0)
    G = XA.T @ XA
    G.flat[::G.shape[0]+1] += n * lam2
    try:
        Ginv = np.linalg.inv(G)
    except np.linalg.LinAlgError:
        Ginv = np.linalg.pinv(G)
    H = XA @ (Ginv @ XA.T)
    h = np.clip(np.diag(H), 0.0, 0.9999)
    Zhat = X @ B
    R = Z - Zhat
    Zhat_alo = Zhat + (h[:, None] / (1 - h[:, None])) * R
    rmse = float(np.sqrt(np.mean((Z - Zhat_alo)**2)))
    return rmse

def score_path_ebic_alo(
    Xs_tr: pd.DataFrame, Ys_tr_scores: pd.DataFrame,
    path: PathResult, l1_ratio: float,
    gamma_ebic: float = 0.5
) -> pd.DataFrame:
    n = len(Xs_tr); p = Xs_tr.shape[1]; r = Ys_tr_scores.shape[1]
    rows = []
    for a_abs, a_frac, B, rss, df in zip(path.alphas_abs, path.alphas_frac, path.coefs, path.rss, path.df_rows):
        ebic = ebic_score(n, r, rss, df, p, gamma=gamma_ebic)
        alo = alo_rmse_active(Xs_tr, Ys_tr_scores, B, alpha=a_abs, l1_ratio=l1_ratio)
        rows.append(dict(alpha_abs=a_abs, alpha_frac=a_frac, df_rows=df,
                         EBIC=ebic, ALO_RMSE=alo, eff_L1=a_abs*l1_ratio))
    df = pd.DataFrame(rows).sort_values(["EBIC", "eff_L1"], ascending=[True, False]).reset_index(drop=True)
    return df

# =====================================================================================
# KKT margin adequacy on PCA scores
# =====================================================================================

@dataclass
class KKTMargin:
    min_margin: float
    threshold: float
    ok: bool
    sigma_hat: float

def kkt_margin_check(
    Xs_tr: pd.DataFrame, Ys_tr_scores: pd.DataFrame,
    B: np.ndarray, alpha: float, l1_ratio: float,
    c: float = 1.5
) -> KKTMargin:
    X = Xs_tr.values; Z = Ys_tr_scores.values
    n, p = X.shape
    R = Z - X @ B
    sigma_hat = float(np.sqrt(np.mean(R**2)))
    S = X.T @ R
    s = np.sqrt((S**2).sum(axis=1)) / n
    rhs = alpha * l1_ratio
    margins = rhs - s
    min_margin = float(margins.min())
    tau = float(c * sigma_hat * np.sqrt(2.0 * np.log(max(p, 2)) / n))
    return KKTMargin(min_margin=min_margin, threshold=tau, ok=(min_margin > tau), sigma_hat=sigma_hat)

# =====================================================================================
# De-biasing (batched) + BH-FDR
# =====================================================================================

def nodewise_precision(X: np.ndarray, alpha_node: float = 1e-3) -> tuple[np.ndarray, np.ndarray]:
    n, p = X.shape
    Theta = np.zeros((p, p), dtype=np.float32)
    tau2 = np.zeros(p, dtype=np.float32)
    for j in range(p):
        mask = np.ones(p, dtype=bool); mask[j] = False
        Xj = X[:, j]; Xm = X[:, mask]
        l = Lasso(alpha=alpha_node, fit_intercept=False, max_iter=3000)
        l.fit(Xm, Xj)
        r = Xj - Xm @ l.coef_.astype(np.float32)
        t2 = float(np.mean(r**2))
        theta_row = np.zeros(p, dtype=np.float32)
        theta_row[j] = 1.0 / max(t2, 1e-12)
        theta_row[mask] = -l.coef_.astype(np.float32) / max(t2, 1e-12)
        Theta[j, :] = theta_row
        tau2[j] = t2
    return Theta, tau2

def debias_coeffs_batched(
    X: np.ndarray, Z: np.ndarray, B_hat: np.ndarray, Theta: np.ndarray,
    batch: int = 512
) -> tuple[np.ndarray, np.ndarray]:
    n = X.shape[0]; r = Z.shape[1]
    E_full = Z - X @ B_hat   # n × r
    sigma2 = np.mean(E_full**2, axis=0)  # length r
    B_tilde = B_hat.copy().astype(np.float32)
    for a in range(0, r, batch):
        b = min(a + batch, r)
        E = E_full[:, a:b]
        XT_E = X.T @ E
        corr = (Theta @ XT_E) / n
        B_tilde[:, a:b] = B_hat[:, a:b] + corr
    return B_tilde, sigma2

def ztests_from_debias_scores(B_tilde: np.ndarray, Theta: np.ndarray, sigma2: np.ndarray, n: int) -> np.ndarray:
    p, r = B_tilde.shape
    var = (Theta.diagonal()[:, None] * sigma2[None, :]) / max(n, 1)
    se = np.sqrt(np.maximum(var, 1e-15))
    Z = B_tilde / se
    return Z

def pvals_from_z(Z: np.ndarray) -> np.ndarray:
    return 2 * (1 - stats.norm.cdf(np.abs(Z)))

def bh_fdr_rowwise(P: np.ndarray, alpha: float = 0.05) -> np.ndarray:
    p, r = P.shape
    sig = np.zeros_like(P, dtype=bool)
    for j in range(p):
        pvals = P[j, :]
        order = np.argsort(pvals)
        pv_sorted = pvals[order]
        m = len(pv_sorted)
        thresh = (np.arange(1, m+1) * alpha) / m
        k = np.where(pv_sorted <= thresh)[0]
        if k.size > 0:
            cutoff = pv_sorted[k.max()]
            sig[j, :] = pvals <= cutoff
    return sig

# =====================================================================================
# Metrics and plotting
# =====================================================================================


def compute_metrics(Y_true: np.ndarray, Y_pred: np.ndarray, df_eff: int) -> dict[str, float]:
    resid = Y_true - Y_pred
    n, q = resid.shape
    rss_total = float(np.sum(resid**2))
    rmse_macro = float(np.sqrt(np.mean(resid**2)))

    sse_vec = np.sum(resid**2, axis=0)
    ss_tot_vec = np.sum((Y_true - Y_true.mean(axis=0))**2, axis=0)
    valid = ss_tot_vec > 1e-12
    r2_vec = np.full(q, np.nan, dtype=np.float64)
    r2_vec[valid] = 1.0 - (sse_vec[valid] / ss_tot_vec[valid])

    aic = 2 * df_eff + n * q * np.log(max(rss_total / (n * q), 1e-15))
    bic = df_eff * np.log(n * q) + n * q * np.log(max(rss_total / (n * q), 1e-15))
    return dict(
        RMSE_macro=rmse_macro,
        R2_macro=float(np.nanmean(r2_vec)) if np.any(valid) else float("nan"),
        R2_median=float(np.nanmedian(r2_vec)) if np.any(valid) else float("nan"),
        R2_used_outputs=int(np.sum(valid)),
        R2_total_outputs=int(q),
        AIC=aic,
        BIC=bic,
    )


def macro_nrmse_with_ref(Y_true: np.ndarray, Y_pred: np.ndarray, Y_ref: np.ndarray, min_range: float = 1e-6) -> tuple[float, dict, np.ndarray]:
    q = Y_true.shape[1]
    rmse = np.sqrt(np.mean((Y_true - Y_pred)**2, axis=0))
    rng = (np.max(Y_ref, axis=0) - np.min(Y_ref, axis=0))
    mask = rng >= min_range
    used = int(np.sum(mask))
    nrmse = np.divide(rmse[mask], rng[mask], out=np.zeros_like(rmse[mask]), where=mask[mask])
    macro = float(np.mean(nrmse)) if used > 0 else float("nan")
    return macro, dict(k_used=used, k_total=q), rmse


def bootstrap_macro_nrmse_ci(
    Y_true: np.ndarray,
    Y_pred: np.ndarray,
    Y_ref: np.ndarray,
    min_range: float = 1e-6,
    n_boot: int = 1000,
    alpha: float = 0.05,
    random_state: int = 123,
) -> dict[str, float]:
    """Row-bootstrap percentile CI for macro nRMSE on a fixed evaluation set."""
    Y_true = np.asarray(Y_true, dtype=np.float64)
    Y_pred = np.asarray(Y_pred, dtype=np.float64)
    Y_ref = np.asarray(Y_ref, dtype=np.float64)

    point, info, rmse_vec = macro_nrmse_with_ref(Y_true, Y_pred, Y_ref, min_range=min_range)
    n = Y_true.shape[0]
    if n <= 0:
        return {
            "point_estimate": float("nan"),
            "ci_lower": float("nan"),
            "ci_upper": float("nan"),
            "bootstrap_mean": float("nan"),
            "bootstrap_std": float("nan"),
            "n_boot": int(n_boot),
            "alpha": float(alpha),
            "k_used": int(info["k_used"]),
            "k_total": int(info["k_total"]),
        }

    rng = np.random.RandomState(random_state)
    boot_stats = np.full(int(n_boot), np.nan, dtype=np.float64)
    for b in range(int(n_boot)):
        idx = rng.randint(0, n, size=n)
        boot_stats[b], _, _ = macro_nrmse_with_ref(
            Y_true[idx, :],
            Y_pred[idx, :],
            Y_ref,
            min_range=min_range,
        )

    valid = np.isfinite(boot_stats)
    if np.any(valid):
        lo = float(np.nanquantile(boot_stats[valid], alpha / 2))
        hi = float(np.nanquantile(boot_stats[valid], 1 - alpha / 2))
        boot_mean = float(np.nanmean(boot_stats[valid]))
        boot_std = float(np.nanstd(boot_stats[valid], ddof=1)) if np.sum(valid) > 1 else 0.0
    else:
        lo = hi = boot_mean = boot_std = float("nan")

    return {
        "point_estimate": float(point),
        "ci_lower": lo,
        "ci_upper": hi,
        "bootstrap_mean": boot_mean,
        "bootstrap_std": boot_std,
        "n_boot": int(n_boot),
        "alpha": float(alpha),
        "k_used": int(info["k_used"]),
        "k_total": int(info["k_total"]),
        "rmse_macro": float(np.nanmean(rmse_vec)) if rmse_vec.size else float("nan"),
    }


def make_null_mean_prediction(Y_train: np.ndarray, n_rows: int) -> np.ndarray:
    """Intercept-only baseline using the train-set mean for each output."""
    Y_train = np.asarray(Y_train, dtype=np.float64)
    mu = np.nanmean(Y_train, axis=0)
    return np.repeat(mu[None, :], int(n_rows), axis=0)


def build_nrmse_summary_rows(
    *,
    stage_label: str,
    evaluation_label: str,
    model_label: str,
    Y_true: np.ndarray,
    Y_pred: np.ndarray,
    Y_ref: np.ndarray,
    min_range: float = 1e-6,
    n_boot: int = 1000,
    alpha: float = 0.05,
    random_state: int = 123,
) -> tuple[dict[str, float], pd.DataFrame]:
    ci = bootstrap_macro_nrmse_ci(
        Y_true=Y_true,
        Y_pred=Y_pred,
        Y_ref=Y_ref,
        min_range=min_range,
        n_boot=n_boot,
        alpha=alpha,
        random_state=random_state,
    )
    row = {
        "stage": stage_label,
        "evaluation_set": evaluation_label,
        "model": model_label,
        "nrmse_macro": ci["point_estimate"],
        "nrmse_ci_lower": ci["ci_lower"],
        "nrmse_ci_upper": ci["ci_upper"],
        "bootstrap_mean": ci["bootstrap_mean"],
        "bootstrap_std": ci["bootstrap_std"],
        "n_boot": ci["n_boot"],
        "alpha": ci["alpha"],
        "k_used_outputs": ci["k_used"],
        "k_total_outputs": ci["k_total"],
        "rmse_macro": ci["rmse_macro"],
        "n_eval_rows": int(np.asarray(Y_true).shape[0]),
    }
    return ci, pd.DataFrame([row])

def save_true_vs_pred_plots(Y_true_df: pd.DataFrame, Y_pred_df: pd.DataFrame, cols: list[str], out_dir: str, prefix: str):
    os.makedirs(out_dir, exist_ok=True)
    for col in cols:
        fig, ax = plt.subplots(figsize=(4.5, 4.0))
        ax.scatter(Y_true_df[col].values, Y_pred_df[col].values, s=6, alpha=0.6)
        ax.set_xlabel("True"); ax.set_ylabel("Predicted"); ax.set_title(f"{prefix} {col}")
        plt.tight_layout()
        fig.savefig(os.path.join(out_dir, f"{prefix}_true_vs_pred__{col}.png"), dpi=140)
        plt.close(fig)

def save_residual_diagnostics(y_true: np.ndarray, y_pred: np.ndarray, X_frame: pd.DataFrame, out_dir: str, prefix: str):
    os.makedirs(out_dir, exist_ok=True)
    resid = y_true - y_pred
    fig, ax = plt.subplots(figsize=(4.5, 4.0))
    ax.scatter(y_pred, resid, s=6, alpha=0.6)
    ax.axhline(0, color="k", lw=1)
    ax.set_title(f"Residuals vs Fitted — {prefix}"); ax.set_xlabel("Fitted"); ax.set_ylabel("Residuals")
    plt.tight_layout(); fig.savefig(os.path.join(out_dir, f"{prefix}_resid_vs_fitted.png"), dpi=140); plt.close(fig)
    fig = plt.figure(figsize=(4.5, 4.0))
    sm.ProbPlot(resid).qqplot(line="45", ax=plt.gca())
    plt.title(f"QQ Plot — {prefix}")
    plt.tight_layout(); fig.savefig(os.path.join(out_dir, f"{prefix}_qq.png"), dpi=140); plt.close(fig)

# =====================================================================================
# Step 1: Hyperparameter search (alpha fractions × l1_ratio list) with PCA Y
# =====================================================================================

@dataclass
class AlphaSelection:
    alpha_frac: float
    alpha_abs: float
    l1_ratio: float
    criterion: str
    table: pd.DataFrame
    kkt: KKTMargin
    x_scaler: StandardScaler
    y_scaler: StandardScaler
    pca: PCA
    subset_index: pd.Index
    kept_outputs: list[str]
    cull_report_csv: str

def run_hparam_search_pca(
    X: pd.DataFrame, Y: pd.DataFrame,
    l1_ratio_list: list[float] = (1.0,),              # exposed knob
    alpha_fracs: list[float] = (0.5, 0.25, 0.1, 0.05, 0.02),
    n_sub: int = 2500,
    random_state: int = 123,
    criterion: str = "EBIC",                          # or "ALO"
    gamma_ebic: float = 0.5,
    max_iter: int = 3000, tol: float = 1e-2,
    selection: str = "random",
    y_pca_variance: float | None = 0.95,
    y_pca_components: int | None = None,
    nzv_std_threshold: float = 1e-8,
    snr_min: float = 0.01,
    cache_dir: str = "model_cache",
    dataset_tag: str | None = None,
    resume: bool = True,
    log_level: int = logging.INFO
) -> AlphaSelection:
    setup_logging(log_level)
    t0 = time.perf_counter()
    logging.info("=== STEP 1: Hyperparameter search (PCA Y) START ===")

    cull = cull_outputs_nzv_snr(X, Y, nzv_std_threshold=nzv_std_threshold, snr_min=snr_min,
                                cache_dir=cache_dir, dataset_tag=dataset_tag, resume=resume)
    Y_kept = Y[cull.kept_outputs]
    pos = _choose_pos_rows(X, n_sub, random_state)
    X_sub = X.iloc[pos]; Y_sub = Y_kept.iloc[pos]

    S = standardize_split_pcaY(X_sub, Y_sub, test_size=0.0, random_state=random_state,
                               y_pca_variance=y_pca_variance, y_pca_components=y_pca_components)
    Xs, Zs = S.Xs_tr, S.Ys_tr_scores

    results = []
    for li, l1r in enumerate(l1_ratio_list, 1):
        logging.info(f"[Grid] l1_ratio={l1r:.3f} ({li}/{len(l1_ratio_list)}) → computing path")
        path = fit_path_by_fractions(Xs, Zs, l1_ratio=l1r, alpha_fracs=alpha_fracs,
                                     max_iter=max_iter, tol=tol, selection=selection)
        table = score_path_ebic_alo(Xs, Zs, path, l1_ratio=l1r, gamma_ebic=gamma_ebic)
        table["l1_ratio"] = l1r
        results.append(table)

    allres = pd.concat(results, ignore_index=True)
    if criterion.upper() == "ALO":
        allres = allres.sort_values(["ALO_RMSE", "eff_L1"], ascending=[True, False]).reset_index(drop=True)
        crit = "ALO"
    else:
        allres = allres.sort_values(["EBIC", "eff_L1"], ascending=[True, False]).reset_index(drop=True)
        crit = "EBIC"

    alpha_frac = float(allres.loc[0, "alpha_frac"])
    l1_ratio  = float(allres.loc[0, "l1_ratio"])
    alpha_abs = float(allres.loc[0, "alpha_abs"])

    path_for_kkt = fit_path_by_fractions(Xs, Zs, l1_ratio=l1_ratio, alpha_fracs=[alpha_frac],
                                         max_iter=max_iter, tol=tol, selection=selection)
    B_star = path_for_kkt.coefs[0]
    kkt = kkt_margin_check(Xs, Zs, B_star, alpha=alpha_abs, l1_ratio=l1_ratio, c=1.5)
    logging.info(f"[α pick] crit={crit}, α_frac={alpha_frac:.4g}, α_abs={alpha_abs:.6g}, l1={l1_ratio:.2f}")
    logging.info(f"[KKT] min_margin={kkt.min_margin:.4g}, τ(n)={kkt.threshold:.4g} → adequate={kkt.ok}")
    logging.info(f"=== STEP 1 DONE in {_sec(time.perf_counter()-t0)} ===")

    sig = dataset_signature(X, Y_kept, dataset_tag)
    key = hashlib.sha1(json.dumps(dict(sig=sig, n_sub=n_sub, fracs=list(alpha_fracs),
                                       l1=list(l1_ratio_list), crit=crit)).encode()).hexdigest()
    csv_out = os.path.join(cache_dir, f"grid_pca__{key}.csv")
    os.makedirs(cache_dir, exist_ok=True)
    allres.to_csv(csv_out, index=False)
    logging.info(f"[grid] saved full grid table → {csv_out}")

    return AlphaSelection(
        alpha_frac=alpha_frac, alpha_abs=alpha_abs, l1_ratio=l1_ratio, criterion=crit,
        table=allres, kkt=kkt, x_scaler=S.x_scaler, y_scaler=S.y_scaler, pca=S.pca,
        subset_index=S.tr_index, kept_outputs=cull.kept_outputs, cull_report_csv=cull.report_csv
    )

# =====================================================================================
# Step 2: Final LASSO on PCA Y with de-biasing, BH-FDR, validation plots/tests
# =====================================================================================

@dataclass
class FinalLassoResult:
    kept_outputs: list[str]
    n_subsample: int
    pca: PCA
    y_scaler: StandardScaler
    B_hat: np.ndarray          # p × r
    Theta: np.ndarray          # p × p
    B_tilde: np.ndarray        # p × r
    sigma2: np.ndarray         # length r
    Z: np.ndarray              # p × r
    P: np.ndarray              # p × r
    significant_mask: np.ndarray  # p × r
    selected_features: list[str]
    metrics: dict[str, float]
    selection_csv: str
    validation_dir: str

def run_final_lasso_pca(
    X: pd.DataFrame, Y: pd.DataFrame,
    X_holdout: pd.DataFrame = None, Y_holdout: pd.DataFrame = None,
    tuned_alpha_frac: float = 0.10, tuned_l1_ratio: float = 1.0,
    n_sub: int = 3000,
    test_size: float = 0.10,
    random_state: int = 123,
    base_max_iter: int = 3000, base_tol: float = 1e-2,
    selection: str = "random",
    y_pca_variance: float | None = 0.95,
    y_pca_components: int | None = None,
    nzv_std_threshold: float = 1e-8,
    snr_min: float = 0.01,
    debias_batch: int = 512,
    fdr_alpha: float = 0.05,
    visuals_outputs_subset: list[str] | None = None,
    out_dir: str = "model_artifacts",
    cache_dir: str = "model_cache", dataset_tag: str | None = None,
    resume: bool = True,
    nrmse_boot_n: int = 1000,
    nrmse_boot_alpha: float = 0.05,
    log_level: int = logging.INFO
) -> FinalLassoResult:
    setup_logging(log_level)
    t0 = time.perf_counter()
    logging.info("=== STEP 2: Final LASSO (PCA Y) START ===")

    cull = cull_outputs_nzv_snr(X, Y, nzv_std_threshold=nzv_std_threshold, snr_min=snr_min,
                                cache_dir=cache_dir, dataset_tag=dataset_tag, resume=resume)
    Y_kept = Y[cull.kept_outputs]

    pos = _choose_pos_rows(X, n_sub, random_state)
    X_sub = X.iloc[pos]; Y_sub = Y_kept.iloc[pos]

    S = standardize_split_pcaY(
        X_sub, Y_sub, test_size=test_size, random_state=random_state,
        y_pca_variance=y_pca_variance, y_pca_components=y_pca_components
    )
    Xs_tr, Xs_te = S.Xs_tr, S.Xs_te
    Zs_tr, Zs_te = S.Ys_tr_scores, S.Ys_te_scores
    pca, ysc = S.pca, S.y_scaler

    alpha_max = compute_alpha_max_mtl_scores(Xs_tr, Zs_tr)
    alpha_abs = (tuned_alpha_frac / max(tuned_l1_ratio, 1e-6)) * alpha_max

    B_hat, info = fit_mtenet_once(
        Xs_tr, Zs_tr, alpha=alpha_abs, l1_ratio=tuned_l1_ratio,
        max_iter=base_max_iter, tol=base_tol, selection=selection, warm_start_coef_=None
    )
    logging.info(f"Final LASSO fit (scores): n_iter={info['n_iter']}, time={_sec(info['time'])}")

    Xn = Xs_tr.values.astype(np.float32)
    Theta, tau2 = nodewise_precision(Xn, alpha_node=max(alpha_abs*0.5, 1e-3))

    B_tilde, sigma2 = debias_coeffs_batched(Xn, Zs_tr.values.astype(np.float32), B_hat, Theta, batch=int(debias_batch))
    Zstats = ztests_from_debias_scores(B_tilde, Theta, sigma2, n=len(Xs_tr))
    Pvals = pvals_from_z(Zstats)

    sigmask = bh_fdr_rowwise(Pvals, alpha=float(fdr_alpha))
    keep_rows = sigmask.any(axis=1)
    selected_features = list(Xs_tr.columns[keep_rows])

    sel_rows = []
    for j, name in enumerate(Xs_tr.columns):
        any_sig = bool(sigmask[j, :].any())
        min_p = float(np.min(Pvals[j, :])) if Pvals.shape[1] > 0 else np.nan
        reason = "kept_BH-FDR" if any_sig else "filtered_not_significant"
        sel_rows.append(dict(feature=name, kept=int(any_sig), min_p=min_p, reason=reason))
    sel_df = pd.DataFrame(sel_rows)
    os.makedirs(out_dir, exist_ok=True)
    selection_csv = os.path.join(out_dir, "final_lasso_selection.csv")
    sel_df.to_csv(selection_csv, index=False)
    logging.info(f"[selection] saved predictor selection CSV → {selection_csv}")

    metrics = {}
    valid_dir = os.path.join(out_dir, "final_lasso_validation")
    os.makedirs(valid_dir, exist_ok=True)
    if Xs_te is not None and Zs_te is not None:
        Zhat_te = Xs_te.values @ B_hat
        Yhat_te_z = pca.inverse_transform(Zhat_te)
        Yhat_te = ysc.inverse_transform(Yhat_te_z)
        Y_true_te = Y.loc[Xs_te.index, cull.kept_outputs].values

        nrmse, info_n, _ = macro_nrmse_with_ref(Y_true_te, Yhat_te, Y_sub.values, min_range=1e-6)
        metr = compute_metrics(Y_true_te, Yhat_te, df_eff=int(np.count_nonzero(B_hat)))
        metr["nRMSE_macro"] = nrmse
        metr["nRMSE_used_outputs"] = info_n["k_used"]
        metr["nRMSE_total_outputs"] = info_n["k_total"]

        screening_ci, screening_row = build_nrmse_summary_rows(
            stage_label="screening",
            evaluation_label="internal_validation_split",
            model_label="screening_lasso",
            Y_true=Y_true_te,
            Y_pred=Yhat_te,
            Y_ref=Y_sub.values,
            min_range=1e-6,
            n_boot=nrmse_boot_n,
            alpha=nrmse_boot_alpha,
            random_state=random_state,
        )
        null_pred_te = make_null_mean_prediction(Y_sub.values, n_rows=Y_true_te.shape[0])
        screening_null_ci, screening_null_row = build_nrmse_summary_rows(
            stage_label="screening",
            evaluation_label="internal_validation_split",
            model_label="null_mean_baseline",
            Y_true=Y_true_te,
            Y_pred=null_pred_te,
            Y_ref=Y_sub.values,
            min_range=1e-6,
            n_boot=nrmse_boot_n,
            alpha=nrmse_boot_alpha,
            random_state=random_state + 1,
        )

        metr["nRMSE_ci_lower"] = screening_ci["ci_lower"]
        metr["nRMSE_ci_upper"] = screening_ci["ci_upper"]
        metr["null_nRMSE_macro"] = screening_null_ci["point_estimate"]
        metr["null_nRMSE_ci_lower"] = screening_null_ci["ci_lower"]
        metr["null_nRMSE_ci_upper"] = screening_null_ci["ci_upper"]
        metrics = metr
        screening_summary_df = pd.concat([screening_row, screening_null_row], ignore_index=True)
        screening_summary_csv = os.path.join(valid_dir, "lasso_internal_nrmse_summary.csv")
        screening_summary_json = os.path.join(valid_dir, "lasso_internal_nrmse_summary.json")
        screening_summary_df.to_csv(screening_summary_csv, index=False)
        with open(screening_summary_json, "w") as f:
            json.dump(screening_summary_df.to_dict(orient="records"), f, indent=2)

        with open(os.path.join(valid_dir, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=2)

        rng = np.random.RandomState(random_state)
        sub_cols_idx = rng.choice(np.arange(Y_true_te.shape[1]), size=min(50, Y_true_te.shape[1]), replace=False)
        ad_pvals = []; bp_pvals = []
        for idx in sub_cols_idx:
            y_t = Y_true_te[:, idx]; y_p = Yhat_te[:, idx]
            resid = y_t - y_p
            try:
                ad_stat, ad_p = normal_ad(resid)
            except Exception:
                ad_p = np.nan
            bp_lm, bp_p, _, _ = het_breuschpagan(resid, sm.add_constant(Xs_te.values, has_constant="add"))
            ad_pvals.append(float(ad_p)); bp_pvals.append(float(bp_p))
        tests = dict(
            normality_AD_pvals=ad_pvals,
            heteroskedasticity_BP_pvals=bp_pvals,
            normality_AD_reject_rate=float(np.mean(np.array(ad_pvals) < 0.05)),
            heteroskedasticity_BP_reject_rate=float(np.mean(np.array(bp_pvals) < 0.05))
        )
        with open(os.path.join(valid_dir, "assumption_tests.json"), "w") as f:
            json.dump(tests, f, indent=2)

        cols = visuals_outputs_subset if visuals_outputs_subset else [Y_kept.columns[i] for i in sub_cols_idx[:12]]
        Y_true_df = pd.DataFrame(Y_true_te, columns=Y_kept.columns, index=Xs_te.index)
        Y_pred_df = pd.DataFrame(Yhat_te, columns=Y_kept.columns, index=Xs_te.index)
        save_true_vs_pred_plots(Y_true_df, Y_pred_df, cols, valid_dir, prefix="lasso")
        if cols:
            c0 = cols[0]
            save_residual_diagnostics(Y_true_df[c0].values, Y_pred_df[c0].values, Xs_te, valid_dir, prefix=f"lasso_{c0}")

    
    # Optional external holdout evaluation (no fitting)
    if X_holdout is not None and Y_holdout is not None:
        X_hold_s = pd.DataFrame(S.x_scaler.transform(X_holdout), index=X_holdout.index, columns=X_holdout.columns)
        Zhat_hold = X_hold_s.values @ B_hat
        Yhat_hold_z = pca.inverse_transform(Zhat_hold)
        Yhat_hold = ysc.inverse_transform(Yhat_hold_z)
        Y_true_hold = Y_holdout.loc[:, cull.kept_outputs].values
        nrmse_h, info_h, _ = macro_nrmse_with_ref(Y_true_hold, Yhat_hold, Y_sub.values, min_range=1e-6)
        metr_h = compute_metrics(Y_true_hold, Yhat_hold, df_eff=int(np.count_nonzero(B_hat)))
        metr_h["nRMSE_macro"] = nrmse_h
        metr_h["nRMSE_used_outputs"] = info_h["k_used"]; metr_h["nRMSE_total_outputs"] = info_h["k_total"]

        holdout_ci, holdout_row = build_nrmse_summary_rows(
            stage_label="screening",
            evaluation_label="external_holdout",
            model_label="screening_lasso",
            Y_true=Y_true_hold,
            Y_pred=Yhat_hold,
            Y_ref=Y_sub.values,
            min_range=1e-6,
            n_boot=nrmse_boot_n,
            alpha=nrmse_boot_alpha,
            random_state=random_state,
        )
        null_pred_hold = make_null_mean_prediction(Y_sub.values, n_rows=Y_true_hold.shape[0])
        holdout_null_ci, holdout_null_row = build_nrmse_summary_rows(
            stage_label="screening",
            evaluation_label="external_holdout",
            model_label="null_mean_baseline",
            Y_true=Y_true_hold,
            Y_pred=null_pred_hold,
            Y_ref=Y_sub.values,
            min_range=1e-6,
            n_boot=nrmse_boot_n,
            alpha=nrmse_boot_alpha,
            random_state=random_state + 1,
        )

        metr_h["nRMSE_ci_lower"] = holdout_ci["ci_lower"]
        metr_h["nRMSE_ci_upper"] = holdout_ci["ci_upper"]
        metr_h["null_nRMSE_macro"] = holdout_null_ci["point_estimate"]
        metr_h["null_nRMSE_ci_lower"] = holdout_null_ci["ci_lower"]
        metr_h["null_nRMSE_ci_upper"] = holdout_null_ci["ci_upper"]

        holdout_summary_df = pd.concat([holdout_row, holdout_null_row], ignore_index=True)
        holdout_summary_csv = os.path.join(valid_dir, "lasso_holdout_nrmse_summary.csv")
        holdout_summary_json = os.path.join(valid_dir, "lasso_holdout_nrmse_summary.json")
        holdout_summary_df.to_csv(holdout_summary_csv, index=False)

        with open(os.path.join(valid_dir, "lasso_holdout_metrics.json"), "w") as f:
            json.dump(metr_h, f, indent=2)
        with open(holdout_summary_json, "w") as f:
            json.dump(holdout_summary_df.to_dict(orient="records"), f, indent=2)
        rng = np.random.RandomState(random_state)
        sub_cols_idx = rng.choice(np.arange(Y_true_hold.shape[1]), size=min(12, Y_true_hold.shape[1]), replace=False)
        cols = [Y_kept.columns[i] for i in sub_cols_idx]
        Y_true_hold_df = pd.DataFrame(Y_true_hold, columns=Y_kept.columns, index=X_holdout.index)
        Y_pred_hold_df = pd.DataFrame(Yhat_hold, columns=Y_kept.columns, index=X_holdout.index)
        save_true_vs_pred_plots(Y_true_hold_df, Y_pred_hold_df, cols, valid_dir, prefix="lasso_ext_holdout")
        if cols:
            c0 = cols[0]
            save_residual_diagnostics(Y_true_hold_df[c0].values, Y_pred_hold_df[c0].values,
                                      X_hold_s, valid_dir, prefix=f"lasso_ext_holdout_{c0}")
    logging.info(f"=== STEP 2 DONE in {_sec(time.perf_counter()-t0)} ===")
        
    return FinalLassoResult(
        kept_outputs=cull.kept_outputs, n_subsample=n_sub, pca=pca, y_scaler=ysc,
        B_hat=B_hat, Theta=Theta, B_tilde=B_tilde, sigma2=sigma2, Z=Zstats, P=Pvals,
        significant_mask=sigmask, selected_features=selected_features, metrics=metrics,
        selection_csv=selection_csv, validation_dir=valid_dir
    )

# =====================================================================================
# Step 3: Final OLS on all outputs with automatic HC3, uncertainty, plots/tests
# =====================================================================================

@dataclass
class OLSResult:
    B: np.ndarray
    XtX_inv: np.ndarray
    mode: str | None
    se: np.ndarray
    p: np.ndarray
    ci_lo: np.ndarray
    ci_hi: np.ndarray
    metrics: dict[str, float]
    coef_summary_csv: str
    validation_dir: str
    export_dir: str
    artifact_manifest_json: str



def export_final_ols_bundle(
    *,
    out_dir: str,
    dataset_tag: str | None,
    all_input_features: list[str],
    selected_features: list[str],
    retained_features: list[str],
    dropped_zero_variance_features: list[str],
    output_names: list[str],
    x_scaler: StandardScaler,
    y_scaler: StandardScaler,
    B_std: np.ndarray,
    XtX_inv: np.ndarray,
    se: np.ndarray,
    pvals: np.ndarray,
    ci_lo: np.ndarray,
    ci_hi: np.ndarray,
    covariance_mode: str | None,
    metrics: dict[str, float],
    coef_summary_csv: str,
    validation_dir: str,
) -> dict[str, str]:
    """Save a reusable final-model bundle for scoring new data."""
    export_dir = os.path.join(out_dir, "final_ols_model")
    os.makedirs(export_dir, exist_ok=True)

    all_input_features = list(all_input_features)
    selected_features = list(selected_features)
    retained_features = list(retained_features)
    dropped_zero_variance_features = list(dropped_zero_variance_features)
    coef_row_names = ["const"] + retained_features
    output_names = list(output_names)

    x_mean = np.asarray(x_scaler.mean_, dtype=np.float64)
    x_std = np.asarray(x_scaler.scale_, dtype=np.float64)
    y_mean = np.asarray(y_scaler.mean_, dtype=np.float64)
    y_std = np.asarray(y_scaler.scale_, dtype=np.float64)

    beta_std = np.asarray(B_std[1:, :], dtype=np.float64)
    intercept_std = np.asarray(B_std[0, :], dtype=np.float64)

    safe_x_std = np.where(np.abs(x_std) < 1e-15, 1.0, x_std)
    beta_raw = (y_std[None, :] * beta_std) / safe_x_std[:, None]
    intercept_raw = y_mean + y_std * intercept_std - (x_mean[:, None] * beta_raw).sum(axis=0)

    all_input_position_map = {name: int(i) for i, name in enumerate(all_input_features)}
    selected_input_position_map = {name: int(i) for i, name in enumerate(selected_features)}
    retained_input_position_map = {name: int(i) for i, name in enumerate(retained_features)}

    all_input_metadata_df = pd.DataFrame(
        {
            "input_position": np.arange(len(all_input_features), dtype=int),
            "input_name": all_input_features,
        }
    )
    all_input_metadata_df["selected_by_lasso"] = all_input_metadata_df["input_name"].isin(selected_features)
    all_input_metadata_df["retained_for_final_ols"] = all_input_metadata_df["input_name"].isin(retained_features)
    all_input_metadata_df["dropped_zero_variance_for_final_ols"] = all_input_metadata_df["input_name"].isin(dropped_zero_variance_features)

    selected_input_metadata_df = pd.DataFrame(
        {
            "selected_input_position": np.arange(len(selected_features), dtype=int),
            "input_name": selected_features,
            "original_input_position": [all_input_position_map[name] for name in selected_features],
            "retained_for_final_ols": [name in retained_input_position_map for name in selected_features],
            "dropped_zero_variance_for_final_ols": [name in dropped_zero_variance_features for name in selected_features],
        }
    )

    retained_input_transform_df = pd.DataFrame(
        {
            "design_input_position": np.arange(len(retained_features), dtype=int),
            "original_input_position": [all_input_position_map[name] for name in retained_features],
            "selected_input_position": [selected_input_position_map.get(name, -1) for name in retained_features],
            "feature_name": retained_features,
            "mean": x_mean,
            "std": x_std,
        }
    )

    output_metadata_df = pd.DataFrame(
        {
            "output_position": np.arange(len(output_names), dtype=int),
            "output_name": output_names,
        }
    )

    output_transform_df = pd.DataFrame(
        {
            "output_position": np.arange(len(output_names), dtype=int),
            "output_name": output_names,
            "mean": y_mean,
            "std": y_std,
        }
    )

    all_input_metadata_csv = os.path.join(export_dir, "all_input_metadata.csv")
    selected_input_metadata_csv = os.path.join(export_dir, "selected_input_metadata.csv")
    output_metadata_csv = os.path.join(export_dir, "output_metadata.csv")
    x_scaler_csv = os.path.join(export_dir, "x_standardization.csv")
    y_scaler_csv = os.path.join(export_dir, "y_standardization.csv")
    coef_std_csv = os.path.join(export_dir, "coef_matrix_standardized.csv")
    coef_raw_csv = os.path.join(export_dir, "coef_matrix_raw_scale.csv")
    coef_std_npz = os.path.join(export_dir, "coef_matrices_and_inference.npz")
    x_scaler_joblib = os.path.join(export_dir, "x_scaler.joblib")
    y_scaler_joblib = os.path.join(export_dir, "y_scaler.joblib")
    bundle_joblib = os.path.join(export_dir, "final_ols_bundle.joblib")
    manifest_json = os.path.join(export_dir, "model_manifest.json")

    all_input_metadata_df.to_csv(all_input_metadata_csv, index=False)
    selected_input_metadata_df.to_csv(selected_input_metadata_csv, index=False)
    output_metadata_df.to_csv(output_metadata_csv, index=False)
    retained_input_transform_df.to_csv(x_scaler_csv, index=False)
    output_transform_df.to_csv(y_scaler_csv, index=False)

    coef_std_df = pd.DataFrame(np.asarray(B_std, dtype=np.float64), index=coef_row_names, columns=output_names)
    coef_std_df.index.name = "coefficient_row_name"
    coef_std_df.to_csv(coef_std_csv)

    coef_raw_df = pd.DataFrame(
        np.vstack([intercept_raw[None, :], beta_raw]),
        index=coef_row_names,
        columns=output_names,
    )
    coef_raw_df.index.name = "coefficient_row_name"
    coef_raw_df.to_csv(coef_raw_csv)

    np.savez_compressed(
        coef_std_npz,
        B_standardized=np.asarray(B_std, dtype=np.float64),
        B_raw_scale=np.vstack([intercept_raw[None, :], beta_raw]).astype(np.float64),
        XtX_inv=np.asarray(XtX_inv, dtype=np.float64),
        se=np.asarray(se, dtype=np.float64),
        pvals=np.asarray(pvals, dtype=np.float64),
        ci_lo=np.asarray(ci_lo, dtype=np.float64),
        ci_hi=np.asarray(ci_hi, dtype=np.float64),
        x_mean=x_mean,
        x_std=x_std,
        y_mean=y_mean,
        y_std=y_std,
    )

    dump(x_scaler, x_scaler_joblib)
    dump(y_scaler, y_scaler_joblib)
    dump(
        {
            "dataset_tag": dataset_tag,
            "all_input_features": all_input_features,
            "selected_features": selected_features,
            "retained_features": retained_features,
            "dropped_zero_variance_features": dropped_zero_variance_features,
            "output_names": output_names,
            "B_standardized": np.asarray(B_std, dtype=np.float64),
            "B_raw_scale": np.vstack([intercept_raw[None, :], beta_raw]).astype(np.float64),
            "XtX_inv": np.asarray(XtX_inv, dtype=np.float64),
            "se": np.asarray(se, dtype=np.float64),
            "pvals": np.asarray(pvals, dtype=np.float64),
            "ci_lo": np.asarray(ci_lo, dtype=np.float64),
            "ci_hi": np.asarray(ci_hi, dtype=np.float64),
            "x_scaler": x_scaler,
            "y_scaler": y_scaler,
            "coef_row_names": coef_row_names,
            "metrics": dict(metrics),
            "covariance_mode": covariance_mode or "classical",
        },
        bundle_joblib,
    )

    manifest = {
        "dataset_tag": dataset_tag,
        "covariance_mode": covariance_mode or "classical",
        "n_all_input_features": int(len(all_input_features)),
        "n_selected_features": int(len(selected_features)),
        "n_retained_features": int(len(retained_features)),
        "n_outputs": int(len(output_names)),
        "all_input_features": all_input_features,
        "selected_features": selected_features,
        "retained_features": retained_features,
        "dropped_zero_variance_features": dropped_zero_variance_features,
        "output_names": output_names,
        "coef_row_names": coef_row_names,
        "all_input_position_map": all_input_position_map,
        "selected_input_position_map": selected_input_position_map,
        "retained_input_position_map": retained_input_position_map,
        "output_position_map": {name: int(i) for i, name in enumerate(output_names)},
        "coef_row_position_map": {name: int(i) for i, name in enumerate(coef_row_names)},
        "prediction_recipe_standardized": {
            "equation": "Y_std = intercept_std + ((X_raw_retained - x_mean) / x_std) @ beta_std; Y_raw = y_mean + y_std * Y_std",
            "intercept_row_name": "const",
        },
        "prediction_recipe_raw_scale": {
            "equation": "Y_raw = intercept_raw + X_raw_retained @ beta_raw",
            "intercept_row_name": "const",
        },
        "files": {
            "all_input_metadata_csv": all_input_metadata_csv,
            "selected_input_metadata_csv": selected_input_metadata_csv,
            "output_metadata_csv": output_metadata_csv,
            "x_standardization_csv": x_scaler_csv,
            "y_standardization_csv": y_scaler_csv,
            "coef_matrix_standardized_csv": coef_std_csv,
            "coef_matrix_raw_scale_csv": coef_raw_csv,
            "coef_matrices_and_inference_npz": coef_std_npz,
            "x_scaler_joblib": x_scaler_joblib,
            "y_scaler_joblib": y_scaler_joblib,
            "bundle_joblib": bundle_joblib,
            "coef_summary_csv": coef_summary_csv,
            "validation_dir": validation_dir,
        },
        "metrics": {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in metrics.items()},
    }
    with open(manifest_json, "w") as f:
        json.dump(manifest, f, indent=2)

    return {"export_dir": export_dir, "artifact_manifest_json": manifest_json}


def run_final_ols(

    X: pd.DataFrame, Y: pd.DataFrame, features: list[str],
    X_holdout: pd.DataFrame, Y_holdout: pd.DataFrame,
    robust_mode: str | None = None,   # None or "HC3"; None => auto via BP screen on TRAIN
    bp_alpha: float = 0.05,
    out_chunk: int = 1000,
    random_state: int = 123,
    nrmse_min_range: float = 1e-6,
    visuals_outputs_subset: list[str] | None = None,
    out_dir: str = "model_artifacts",
    cache_dir: str = "model_cache", dataset_tag: str | None = None,
    resume: bool = True,
    nrmse_boot_n: int = 1000,
    nrmse_boot_alpha: float = 0.05,
    log_level: int = logging.INFO
) -> OLSResult:
    """
    Final OLS on standardized TRAIN X and standardized TRAIN Y, evaluated on the
    external HOLDOUT after inverse-transforming predictions back to the original Y scale.

    This matches the intended workflow more closely and is much more numerically stable
    than solving the multi-output system directly on raw-scale float32 arrays.
    """
    setup_logging(log_level)
    t0 = time.perf_counter()
    logging.info("=== STEP 3: Final OLS START (external 10% holdout for metrics) ===")

    # Align rows defensively.
    X, Y = X.align(Y, join="inner", axis=0)
    X_holdout, Y_holdout = X_holdout.align(Y_holdout, join="inner", axis=0)

    # Use the selected features if supplied; otherwise fall back to all predictors.
    feat_list = list(X.columns) if len(features) == 0 else [f for f in features if f in X.columns]
    if len(feat_list) == 0:
        raise ValueError("No valid features were supplied to run_final_ols().")

    X_sel_tr = X.loc[:, feat_list].copy()
    X_sel_te = X_holdout.loc[:, feat_list].copy()

    # Drop train-side zero-variance predictors to avoid singular design artifacts.
    x_std = X_sel_tr.std(axis=0, ddof=0)
    keep_x = x_std > 1e-12
    dropped_x = X_sel_tr.columns[~keep_x].tolist()
    if dropped_x:
        logging.info(f"[OLS] dropping {len(dropped_x)} zero-variance predictors on TRAIN.")
    X_sel_tr = X_sel_tr.loc[:, keep_x]
    X_sel_te = X_sel_te.loc[:, keep_x]
    feat_names = ["const"] + list(X_sel_tr.columns)

    # Standardize on TRAIN only and keep float64 precision end to end.
    xsc = StandardScaler(with_mean=True, with_std=True)
    ysc = StandardScaler(with_mean=True, with_std=True)

    Xs_tr = xsc.fit_transform(X_sel_tr).astype(np.float64)
    Xs_te = xsc.transform(X_sel_te).astype(np.float64)
    Ys_tr = ysc.fit_transform(Y).astype(np.float64)

    Xc_tr = sm.add_constant(Xs_tr, has_constant="add")
    Xc_te = sm.add_constant(Xs_te, has_constant="add")

    Y_tr_mat = Y.values.astype(np.float64)
    Y_te_mat = Y_holdout.values.astype(np.float64)
    n_tr, p1 = Xc_tr.shape
    q = Ys_tr.shape[1]

    # Closed-form OLS on standardized scale.
    XtX = Xc_tr.T @ Xc_tr
    XtX_inv = np.linalg.pinv(XtX)
    B_std = XtX_inv @ (Xc_tr.T @ Ys_tr)   # (p+1) × q

    # TRAIN residuals on standardized scale (for robust-mode decision/inference).
    Yhat_tr_std = Xc_tr @ B_std
    resid_tr_std = Ys_tr - Yhat_tr_std

    decided_mode = robust_mode
    if decided_mode is None:
        rng = np.random.RandomState(random_state)
        sample_cols = list(rng.choice(list(range(q)), size=min(50, q), replace=False))
        pvals_bp = []
        for j in sample_cols:
            try:
                lm, lm_p, _, _ = het_breuschpagan(resid_tr_std[:, j], Xc_tr)
            except Exception:
                lm_p = np.nan
            if np.isfinite(lm_p):
                pvals_bp.append(float(lm_p))
        if len(pvals_bp) == 0:
            decided_mode = "HC3"
            logging.info("[OLS-BP screen] all BP p-values were non-finite; defaulting to HC3.")
        else:
            pvals_bp = np.asarray(pvals_bp, dtype=np.float64)
            share_rej = float(np.mean(pvals_bp < bp_alpha))
            med_p = float(np.nanmedian(pvals_bp))
            decided_mode = "HC3" if (share_rej >= 0.20 or med_p < bp_alpha) else None
            logging.info(
                f"[OLS-BP screen] share_rejected={share_rej:.2f}, median p={med_p:.3g} "
                f"→ covariance mode={decided_mode or 'classical'}"
            )

    df = max(n_tr - p1, 1)
    se = np.zeros((p1, q), dtype=np.float64)
    stat = np.zeros((p1, q), dtype=np.float64)
    pvals = np.zeros((p1, q), dtype=np.float64)
    ci_lo = np.zeros((p1, q), dtype=np.float64)
    ci_hi = np.zeros((p1, q), dtype=np.float64)

    if decided_mode is None:
        sse = np.sum(resid_tr_std**2, axis=0)
        sigma2 = sse / df
        diag_XtX_inv = np.diag(XtX_inv)
        for k in range(q):
            se[:, k] = np.sqrt(np.maximum(sigma2[k], 1e-12) * diag_XtX_inv)
        stat[:, :] = B_std / np.maximum(se, 1e-15)
        pvals[:, :] = 2 * (1 - stats.t.cdf(np.abs(stat), df))
        crit = stats.t.ppf(1 - 0.05 / 2, df)
        ci_lo[:, :] = B_std - crit * se
        ci_hi[:, :] = B_std + crit * se
    else:
        H_diag = np.sum((Xc_tr @ XtX_inv) * Xc_tr, axis=1)
        adj2 = (1.0 / np.maximum(1 - H_diag, 1e-8)) ** 2
        for a in range(0, q, out_chunk):
            b = min(a + out_chunk, q)
            Eb = resid_tr_std[:, a:b]
            W = (Eb ** 2) * adj2[:, None]
            for kk in range(Eb.shape[1]):
                S = (Xc_tr.T * W[:, kk]) @ Xc_tr
                V = XtX_inv @ S @ XtX_inv
                se[:, a + kk] = np.sqrt(np.maximum(np.diag(V), 1e-15))
            stat[:, a:b] = B_std[:, a:b] / np.maximum(se[:, a:b], 1e-15)
            pvals[:, a:b] = 2 * (1 - stats.norm.cdf(np.abs(stat[:, a:b])))
            crit = stats.norm.ppf(1 - 0.05 / 2)
            ci_lo[:, a:b] = B_std[:, a:b] - crit * se[:, a:b]
            ci_hi[:, a:b] = B_std[:, a:b] + crit * se[:, a:b]

    # Holdout prediction: standardized -> inverse transform to original Y units.
    Yhat_te_std = Xc_te @ B_std
    Yhat_te = ysc.inverse_transform(Yhat_te_std)

    nrmse, info_n, _ = macro_nrmse_with_ref(Y_te_mat, Yhat_te, Y_tr_mat, min_range=nrmse_min_range)
    metrics = compute_metrics(Y_te_mat, Yhat_te, df_eff=p1 * q)
    metrics["nRMSE_macro"] = nrmse
    metrics["nRMSE_used_outputs"] = info_n["k_used"]
    metrics["nRMSE_total_outputs"] = info_n["k_total"]

    final_ci, final_row = build_nrmse_summary_rows(
        stage_label="final",
        evaluation_label="external_holdout",
        model_label="final_ols",
        Y_true=Y_te_mat,
        Y_pred=Yhat_te,
        Y_ref=Y_tr_mat,
        min_range=nrmse_min_range,
        n_boot=nrmse_boot_n,
        alpha=nrmse_boot_alpha,
        random_state=random_state,
    )
    null_pred_te = make_null_mean_prediction(Y_tr_mat, n_rows=Y_te_mat.shape[0])
    null_ci, null_row = build_nrmse_summary_rows(
        stage_label="final",
        evaluation_label="external_holdout",
        model_label="null_mean_baseline",
        Y_true=Y_te_mat,
        Y_pred=null_pred_te,
        Y_ref=Y_tr_mat,
        min_range=nrmse_min_range,
        n_boot=nrmse_boot_n,
        alpha=nrmse_boot_alpha,
        random_state=random_state + 1,
    )
    metrics["nRMSE_ci_lower"] = final_ci["ci_lower"]
    metrics["nRMSE_ci_upper"] = final_ci["ci_upper"]
    metrics["null_nRMSE_macro"] = null_ci["point_estimate"]
    metrics["null_nRMSE_ci_lower"] = null_ci["ci_lower"]
    metrics["null_nRMSE_ci_upper"] = null_ci["ci_upper"]

    os.makedirs(out_dir, exist_ok=True)
    coef_rows = []
    for i, fname in enumerate(feat_names):
        pv = pvals[i, :]
        se_i = se[i, :]
        ci_lo_i = ci_lo[i, :]
        ci_hi_i = ci_hi[i, :]
        coef_rows.append(
            dict(
                feature=fname,
                mean_p=float(np.nanmean(pv)),
                frac_sig=float(np.nanmean(pv < 0.05)),
                mean_se=float(np.nanmean(se_i)),
                mean_ci_lo=float(np.nanmean(ci_lo_i)),
                mean_ci_hi=float(np.nanmean(ci_hi_i)),
            )
        )
    coef_df = pd.DataFrame(coef_rows)
    coef_csv = os.path.join(out_dir, "final_ols_coef_summary.csv")
    coef_df.to_csv(coef_csv, index=False)
    logging.info(f"[OLS] saved coefficient summary → {coef_csv}")

    valid_dir = os.path.join(out_dir, "final_ols_validation")
    os.makedirs(valid_dir, exist_ok=True)

    if visuals_outputs_subset:
        sub_cols = [c for c in visuals_outputs_subset if c in Y_holdout.columns]
    else:
        sub_cols = list(Y_holdout.columns[: min(12, Y_holdout.shape[1])])

    Y_true_df = pd.DataFrame(Y_te_mat, columns=Y.columns, index=Y_holdout.index)
    Y_pred_df = pd.DataFrame(Yhat_te, columns=Y.columns, index=Y_holdout.index)
    save_true_vs_pred_plots(Y_true_df, Y_pred_df, sub_cols, valid_dir, prefix="ols_holdout")

    if len(sub_cols) > 0:
        c0 = sub_cols[0]
        save_residual_diagnostics(
            Y_true_df[c0].values,
            Y_pred_df[c0].values,
            pd.DataFrame(Xc_te, index=Y_holdout.index),
            valid_dir,
            prefix=f"ols_holdout_{c0}",
        )

    ad_pvals, bp_pvals = [], []
    for col in sub_cols:
        resid_c = Y_true_df[col].values - Y_pred_df[col].values
        try:
            _, ad_p = normal_ad(resid_c)
        except Exception:
            ad_p = np.nan
        try:
            _, lm_p, _, _ = het_breuschpagan(resid_c, Xc_te)
        except Exception:
            lm_p = np.nan
        ad_pvals.append(float(ad_p) if np.isfinite(ad_p) else np.nan)
        bp_pvals.append(float(lm_p) if np.isfinite(lm_p) else np.nan)

    tests = dict(
        normality_AD_pvals=ad_pvals,
        heteroskedasticity_BP_pvals=bp_pvals,
        normality_AD_reject_rate=float(np.nanmean(np.array(ad_pvals) < 0.05)),
        heteroskedasticity_BP_reject_rate=float(np.nanmean(np.array(bp_pvals) < 0.05)),
        covariance_mode=decided_mode or "classical",
    )
    nrmse_summary_df = pd.concat([final_row, null_row], ignore_index=True)
    nrmse_summary_csv = os.path.join(valid_dir, "nrmse_summary.csv")
    nrmse_summary_json = os.path.join(valid_dir, "nrmse_summary.json")
    nrmse_summary_df.to_csv(nrmse_summary_csv, index=False)
    with open(nrmse_summary_json, "w") as f:
        json.dump(nrmse_summary_df.to_dict(orient="records"), f, indent=2)

    with open(os.path.join(valid_dir, "assumption_tests.json"), "w") as f:
        json.dump(tests, f, indent=2)
    with open(os.path.join(valid_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f, indent=2)

    export_paths = export_final_ols_bundle(
        out_dir=out_dir,
        dataset_tag=dataset_tag,
        all_input_features=list(X.columns),
        selected_features=feat_list,
        retained_features=list(X_sel_tr.columns),
        dropped_zero_variance_features=dropped_x,
        output_names=list(Y.columns),
        x_scaler=xsc,
        y_scaler=ysc,
        B_std=B_std,
        XtX_inv=XtX_inv,
        se=se,
        pvals=pvals,
        ci_lo=ci_lo,
        ci_hi=ci_hi,
        covariance_mode=decided_mode,
        metrics=metrics,
        coef_summary_csv=coef_csv,
        validation_dir=valid_dir,
    )

    logging.info(f"[OLS] exported reusable model bundle → {export_paths['export_dir']}")
    logging.info(f"=== STEP 3 DONE in {time.perf_counter()-t0:,.2f}s ===")
    return OLSResult(
        B=B_std,
        XtX_inv=XtX_inv,
        mode=decided_mode,
        se=se,
        p=pvals,
        ci_lo=ci_lo,
        ci_hi=ci_hi,
        metrics=metrics,
        coef_summary_csv=coef_csv,
        validation_dir=valid_dir,
        export_dir=export_paths["export_dir"],
        artifact_manifest_json=export_paths["artifact_manifest_json"],
    )

# Example driver (commented): GLOBAL holdout + three stages
# X = pd.read_parquet("X.parquet"); Y = pd.read_parquet("Y.parquet")
# X_tr, Y_tr, X_ho, Y_ho, _, _ = make_global_holdout_split(X, Y, holdout_frac=0.10, random_state=123)
# sel = run_hparam_search_pca(X_tr, Y_tr, l1_ratio_list=(1.0, 0.95, 0.9), alpha_fracs=(0.5, 0.25, 0.1, 0.05, 0.02),
#                             n_sub=2000, random_state=123, criterion="EBIC", y_pca_variance=0.95, tol=1e-2,
#                             selection="random", nzv_std_threshold=1e-8, snr_min=1e-2,
#                             cache_dir="model_cache", dataset_tag="demo", resume=True)
# fl = run_final_lasso_pca(X_tr, Y_tr, X_holdout=X_ho, Y_holdout=Y_ho,
#                          tuned_alpha_frac=sel.alpha_frac, tuned_l1_ratio=sel.l1_ratio,
#                          n_sub=3000, test_size=0.10, random_state=123, base_tol=1e-2, selection="random",
#                          y_pca_variance=0.95, nzv_std_threshold=1e-8, snr_min=1e-2,
#                          debias_batch=512, fdr_alpha=0.05, out_dir="model_artifacts",
#                          cache_dir="model_cache", dataset_tag="demo", resume=True)
# ols = run_final_ols(X_tr, Y_tr, features=fl.selected_features, X_holdout=X_ho, Y_holdout=Y_ho,
#                     robust_mode=None, bp_alpha=0.05, out_dir="model_artifacts",
#                     cache_dir="model_cache", dataset_tag="demo", resume=True)


In [2]:
def get_AFSC(scenario):
    AFSC = 1 if scenario.split('_')[0][4:] == 'on' else 0
    return(AFSC)

In [3]:
def get_UAEORO(scenario):
    UAEORO = 1 if scenario.split('_')[1][6:] == 'on' else 0
    return(UAEORO)

In [5]:
X = pd.read_parquet("sa_068.null_010.X.nl.parquet")
Y = pd.read_parquet("sa_068.null_010.Y.parquet")

print("raw X shape:", X.shape)
print("raw Y shape:", Y.shape)


raw X shape: (20000, 352)
raw Y shape: (20000, 23495)


In [6]:
def _parse_on_off_flag(label: str, token_index: int, prefix_len: int) -> int:
    parts = str(label).split("_")
    value = parts[token_index][prefix_len:]
    return 1 if value == "on" else 0

# Add scenario booleans before moving scenario into the index.
if "scenario" in X.columns:
    scenario_series = X["scenario"].astype(str)
else:
    scenario_series = pd.Series(X.index.get_level_values("scenario"), index=X.index).astype(str)

X = X.copy()
X["AFSC"] = scenario_series.apply(lambda s: _parse_on_off_flag(s, token_index=0, prefix_len=4)).astype(np.int8)
X["UAEORO"] = scenario_series.apply(lambda s: _parse_on_off_flag(s, token_index=1, prefix_len=6)).astype(np.int8)

if {"run_id", "scenario"}.issubset(X.columns):
    X = X.set_index(["run_id", "scenario"])

if {"run_id", "scenario"}.issubset(Y.columns):
    Y = Y.set_index(["run_id", "scenario"])

# Row alignment is mandatory before any split or model fit.
X, Y = X.align(Y, join="inner", axis=0)
X = X.sort_index()
Y = Y.sort_index()

print("aligned X shape:", X.shape)
print("aligned Y shape:", Y.shape)


aligned X shape: (20000, 352)
aligned Y shape: (20000, 23495)


In [7]:
variables = [
    "OI.annual total HCBN prodn",
    "OI.annual HCBN prodn by product[gasoline]",
    "OI.annual HCBN prodn by product[diesel]",
    "OI.annual HCBN prodn by product[JetFuel]",
    "OI.annual HCBN prodn by product[AllOther]",
    "OI.composite HCBN MFSP[NE]",
    "OI.composite HCBN MFSP[A]",
    "OI.composite HCBN MFSP[SE]",
    "OI.composite HCBN MFSP[DS]",
    "OI.composite HCBN MFSP[CB]",
    "OI.composite HCBN MFSP[LS]",
    "OI.composite HCBN MFSP[NP]",
    "OI.composite HCBN MFSP[SP]",
    "OI.composite HCBN MFSP[M]",
    "OI.composite HCBN MFSP[P]",
]

years = list(range(2015, 2052))
plot_target_cols = [f"{variable}_{year}" for variable in variables for year in years]

existing_plot_target_cols = [c for c in plot_target_cols if c in Y.columns]
missing_plot_target_cols = [c for c in plot_target_cols if c not in Y.columns]

print("full output count available for modeling:", Y.shape[1])
print("diagnostic plot outputs requested:", len(plot_target_cols))
print("diagnostic plot outputs found:", len(existing_plot_target_cols))
print("diagnostic plot outputs missing:", len(missing_plot_target_cols))


full output count available for modeling: 23495
diagnostic plot outputs requested: 555
diagnostic plot outputs found: 555
diagnostic plot outputs missing: 0


In [8]:
# Keep only numeric columns and preserve float64 precision throughout the workflow.
X = X.apply(pd.to_numeric, errors="raise").astype(np.float64)
Y = Y.apply(pd.to_numeric, errors="raise").astype(np.float64)

# Deterministic, scenario-stratified global 10% holdout when possible.
if isinstance(X.index, pd.MultiIndex) and "scenario" in X.index.names:
    scenario_labels = X.index.get_level_values("scenario")
else:
    scenario_labels = None

if scenario_labels is not None and scenario_labels.nunique() > 1:
    tr_idx, ho_idx = train_test_split(
        np.arange(len(X)),
        test_size=0.10,
        random_state=123,
        stratify=scenario_labels,
    )
    X_tr, X_ho = X.iloc[np.sort(tr_idx)], X.iloc[np.sort(ho_idx)]
    Y_tr, Y_ho = Y.iloc[np.sort(tr_idx)], Y.iloc[np.sort(ho_idx)]
else:
    X_tr, Y_tr, X_ho, Y_ho, _, _ = make_global_holdout_split(X, Y, holdout_frac=0.10, random_state=123)

print("X_tr:", X_tr.shape, "Y_tr:", Y_tr.shape)
print("X_ho:", X_ho.shape, "Y_ho:", Y_ho.shape)


X_tr: (18000, 352) Y_tr: (18000, 23495)
X_ho: (2000, 352) Y_ho: (2000, 23495)


In [9]:
# 1) Hyperparameter search (train only, no cache reuse while debugging)
sel = run_hparam_search_pca(
    X_tr,
    Y_tr,
    l1_ratio_list=(1.0, 0.95, 0.9),
    alpha_fracs=(0.75, 0.5, 0.25, 0.10, 0.05, 0.02, 0.01),
    n_sub=2000,
    random_state=123,
    criterion="EBIC",
    y_pca_variance=0.95,
    max_iter=10000,
    tol=1e-4,
    selection="cyclic",
    nzv_std_threshold=1e-8,
    snr_min=1e-2,
    cache_dir="model_cache",
    dataset_tag="bsm_reduced_form_targeted_outputs_v4",
    resume=False,
)

sel.table.head(10)


10:43:52 | INFO | === STEP 1: Hyperparameter search (PCA Y) START ===
10:44:06 | INFO | [cull] saved report → model_cache/filt_outputs__c42d8dd143f9e985161eb50f9594d4d2d3ffe098.csv | kept=9782 culled=13713
10:44:47 | INFO | [Grid] l1_ratio=1.000 (1/3) → computing path
10:44:52 | INFO | [PATH] α_frac=0.75, α_abs=21.4057, df_rows=8, rss=1.622e+07, time=4.83s
10:45:00 | INFO | [PATH] α_frac=0.5, α_abs=14.2705, df_rows=21, rss=1.175e+07, time=7.57s
10:45:08 | INFO | [PATH] α_frac=0.25, α_abs=7.13524, df_rows=45, rss=6.328e+06, time=8.45s
10:45:18 | INFO | [PATH] α_frac=0.1, α_abs=2.8541, df_rows=81, rss=3.973e+06, time=10.15s
10:45:28 | INFO | [PATH] α_frac=0.05, α_abs=1.42705, df_rows=112, rss=3.328e+06, time=9.63s
10:45:48 | INFO | [PATH] α_frac=0.02, α_abs=0.57082, df_rows=178, rss=2.753e+06, time=19.91s
10:46:19 | INFO | [PATH] α_frac=0.01, α_abs=0.28541, df_rows=214, rss=2.281e+06, time=30.63s
10:46:19 | INFO | [Grid] l1_ratio=0.950 (2/3) → computing path
10:46:20 | INFO | [PATH] α_fr

,alpha_abs,alpha_frac,df_rows,EBIC,ALO_RMSE,eff_L1,l1_ratio
0,2.854098,0.10,81,605600.346434,3.383160,2.854098,1.00
1,1.427049,0.05,112,620598.626076,3.383160,1.427049,1.00
2,7.135244,0.25,45,627648.486694,3.383160,7.135244,1.00
3,0.570820,0.02,178,680988.871768,3.383160,0.570820,1.00
4,14.270488,0.50,21,689220.297805,3.383160,14.270488,1.00
5,0.285410,0.01,214,701117.730436,3.383160,0.285410,1.00
6,21.405731,0.75,8,720706.846333,3.383160,21.405731,1.00
7,22.532349,0.75,37,764994.769478,3.952241,21.405731,0.95
8,23.784146,0.75,43,775389.986545,4.083614,21.405731,0.90
9,3.004313,0.10,210,788355.980393,3.875635,2.854098,0.95


In [10]:
# 2) Final de-biased LASSO on train only; external holdout only for evaluation
fl = run_final_lasso_pca(
    X_tr,
    Y_tr,
    X_holdout=X_ho,
    Y_holdout=Y_ho,
    tuned_alpha_frac=sel.alpha_frac,
    tuned_l1_ratio=sel.l1_ratio,
    n_sub=3000,
    test_size=0.10,
    random_state=123,
    base_max_iter=10000,
    base_tol=1e-4,
    selection="cyclic",
    y_pca_variance=0.95,
    nzv_std_threshold=1e-8,
    snr_min=1e-2,
    debias_batch=512,
    fdr_alpha=0.05,
    visuals_outputs_subset=existing_plot_target_cols[:12],
    out_dir="model_artifacts",
    cache_dir="model_cache",
    dataset_tag="bsm_reduced_form_full_outputs_v5",
    resume=False,
)

print("selected feature count:", len(fl.selected_features))
print(fl.selected_features[:25])


10:48:23 | INFO | === STEP 2: Final LASSO (PCA Y) START ===
10:48:36 | INFO | [cull] saved report → model_cache/filt_outputs__86d3cf003bf549926d67de782e0e4d99c919caf9.csv | kept=9782 culled=13713
10:49:42 | INFO | Final LASSO fit (scores): n_iter=669, time=22.77s
10:49:58 | INFO | [selection] saved predictor selection CSV → model_artifacts/final_lasso_selection.csv
10:55:01 | INFO | === STEP 2 DONE in 398.17s ===


selected feature count: 346
['AHC.Debt Interest Rate as pct[HEFA]', 'AHC.FCI  sensi multiplier[HEFA]', 'AHC.FCI  sensi multiplier[HTL]', 'AHC.Mature Industry Rate of Return as PCT[HEFA]', 'AHC.Mature Industry Rate of Return as PCT[HTL]', 'AHC.ORNOOC sensi multiplier[HTL]', 'AHC.PY sensi multiplier[HEFA]', 'AHC.PY sensi multiplier[HTL]', 'AHC.initial indices of Commercial Maturity[HEFA]', 'AHC.initial indices of Commercial Maturity[HTL]', 'CHC.Debt Interest Rate as pct[Thermochem]', 'CHC.FCI  sensi multiplier[Biochem]', 'CHC.FCI  sensi multiplier[Brownfield]', 'CHC.FCI  sensi multiplier[Thermochem]', 'CHC.Mature Industry Rate of Return as PCT[Biochem]', 'CHC.Mature Industry Rate of Return as PCT[Brownfield]', 'CHC.Mature Industry Rate of Return as PCT[Thermochem]', 'CHC.ORNOOC sensi multiplier[Thermochem]', 'CHC.PY sensi multiplier[Biochem]', 'CHC.PY sensi multiplier[Brownfield]', 'CHC.PY sensi multiplier[Thermochem]', 'CHC.initial indices of Commercial Maturity[Biochem]', 'CHC.initial 

### Final-input structure summary after de-biased LASSO

In [11]:
import re

import pandas as pd

# Expand this set if you add more transformation generators later.
KNOWN_TRANSFORMATIONS = {
    "quadratic",
    "inverse",
    "log",
    "log1p",
    "sqrt",
    "square",
    "cubic",
    "cube",
    "reciprocal",
    "exp",
    "abs",
}

def _strip_outer_brackets(text):
    text = str(text).strip()
    if text.startswith("[") and text.endswith("]"):
        return text[1:-1].strip()
    return text

def _split_interaction(name):
    """
    Split an interaction term into exactly two components.
    Assumes '*' is only used to denote second-order interactions.
    """
    parts = [part.strip() for part in str(name).split("*")]
    if len(parts) == 2 and all(parts):
        return parts[0], parts[1]
    return None, None

def _split_transformation(name, known_transformations=None):
    """
    Split a nonlinear transformed feature into (base, transformation)
    using the final underscore-delimited suffix.
    """
    if known_transformations is None:
        known_transformations = KNOWN_TRANSFORMATIONS

    name = str(name).strip()
    if "_" not in name:
        return None, None

    base, suffix = name.rsplit("_", 1)
    suffix = suffix.strip()

    if suffix in known_transformations and base.strip():
        return base.strip(), suffix

    return None, None

def parse_selected_input_structure(feature_names, known_transformations=None):
    if known_transformations is None:
        known_transformations = KNOWN_TRANSFORMATIONS

    records = []

    for feature_name in feature_names:
        name = str(feature_name).strip()

        # 1) second-order interaction
        if "*" in name:
            left, right = _split_interaction(name)
            if left is not None:
                feature_type = "second_order"
                base_variable = None
                component_1 = left
                component_2 = right
                transformation = None
            else:
                # fallback for malformed names containing '*'
                feature_type = "unparsed"
                base_variable = name
                component_1 = name
                component_2 = None
                transformation = None

        # 2) nonlinear transformation
        else:
            base, transform = _split_transformation(
                name,
                known_transformations=known_transformations,
            )

            if base is not None:
                feature_type = "nonlinear_transformation"
                base_variable = _strip_outer_brackets(base)
                component_1 = base
                component_2 = None
                transformation = transform
            else:
                # 3) first-order term
                feature_type = "first_order"
                base_variable = _strip_outer_brackets(name)
                component_1 = name
                component_2 = None
                transformation = None

        records.append(
            {
                "input_name": name,
                "input_type": feature_type,
                "base_variable": base_variable,
                "component_1": component_1,
                "component_2": component_2,
                "transformation": transformation,
            }
        )

    parsed_df = pd.DataFrame(records)

    type_summary_df = (
        parsed_df["input_type"]
        .value_counts(dropna=False)
        .rename_axis("input_type")
        .reset_index(name="n_inputs")
        .sort_values("input_type")
        .reset_index(drop=True)
    )

    transformation_summary_df = (
        parsed_df.loc[parsed_df["input_type"] == "nonlinear_transformation", "transformation"]
        .value_counts(dropna=False)
        .rename_axis("transformation")
        .reset_index(name="n_inputs")
        .sort_values("transformation")
        .reset_index(drop=True)
    )

    return parsed_df, type_summary_df, transformation_summary_df

selected_input_structure_df, selected_input_type_summary_df, selected_input_transformation_summary_df = (
    parse_selected_input_structure(fl.selected_features)
)

print("Selected input structure summary:")
display(selected_input_type_summary_df)

print("Selected nonlinear transformation summary:")
display(selected_input_transformation_summary_df)

# selected_input_structure_df.head(20)

Selected input structure summary:


,input_type,n_inputs
0,first_order,62
1,nonlinear_transformation,40
2,second_order,244


Selected nonlinear transformation summary:


,transformation,n_inputs
0,inverse,2
1,quadratic,38


In [12]:
_, selected_input_type_summary_df_X, selected_input_transformation_summary_df = (
    parse_selected_input_structure(X.columns)
)
selected_input_type_summary_df_X

,input_type,n_inputs
0,first_order,63
1,nonlinear_transformation,41
2,second_order,248


In [13]:
if len(fl.selected_features) == 0:
    raise RuntimeError(
        "The de-biased LASSO stage selected zero predictors. "
        "Inspect the hyperparameter grid and the cull report before proceeding to OLS."
    )


In [ ]:
ols = run_final_ols(
    X_tr,
    Y_tr,
    features=fl.selected_features,
    X_holdout=X_ho,
    Y_holdout=Y_ho,
    robust_mode=None,
    bp_alpha=0.05,
    visuals_outputs_subset=existing_plot_target_cols[:12],
    out_dir="model_artifacts",
    cache_dir="model_cache",
    dataset_tag="bsm_reduced_form_full_outputs_v5",
    resume=False,
)

ols.metrics


print('Export directory:', ols.export_dir)
print('Manifest:', ols.artifact_manifest_json)


10:55:01 | INFO | === STEP 3: Final OLS START (external 10% holdout for metrics) ===
/opt/miniconda3/envs/bsm_sens_fy23/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1870: RuntimeWarning: invalid value encountered in scalar divide
  return self.mse_model/self.mse_resid
/opt/miniconda3/envs/bsm_sens_fy23/lib/python3.9/site-packages/statsmodels/regression/linear_model.py:1781: RuntimeWarning: invalid value encountered in scalar divide
  return 1 - self.ssr/self.centered_tss
10:56:25 | INFO | [OLS-BP screen] share_rejected=0.96, median p=0 → covariance mode=HC3


In [ ]:
manifest = json.load(open(ols.artifact_manifest_json))
manifest["files"]


### Debug notes baked into this notebook

This version fixes several concrete workflow defects:

1. `Y` is now explicitly subset to the requested target outputs. In the original notebook, `cols` was defined but never applied, so the model was fitting **all** outputs in the parquet file.
2. `X` and `Y` are aligned on the same rows before any split or fit.
3. The holdout split is scenario-stratified when possible.
4. Debug runs disable cache reuse to avoid stale cull/grid artifacts after preprocessing changes.
5. The final OLS stage now fits on standardized train `X` and standardized train `Y`, then inverse-transforms holdout predictions back to original output units for metrics and plots.
6. Macro `R^2` is now computed across outputs correctly, with zero-variance outputs excluded from the macro average rather than silently collapsing the statistic.

7. The final OLS stage now exports a reusable model bundle with scaler vectors, standardized and raw-scale coefficient matrices, and explicit input/output position maps.
8. The scenario-stratified split cell now has the required `train_test_split` import in notebook scope.


### Post-fit validation, 2050 summaries, trajectory overlays, and parquet exports

These cells:
1. validate the exported coefficient matrices and standardization vectors,
2. reconstruct holdout predictions from the exported model bundle,
3. summarize the 2050 outputs in `variables` for known vs predicted holdout values,
4. plot trajectory overlays for each variable across all available years, and
5. save the coefficient matrices and transform vectors as compressed parquet files.


In [ ]:

from pathlib import Path

postfit_dir = Path(ols.export_dir) / "postfit_diagnostics"
postfit_dir.mkdir(parents=True, exist_ok=True)

manifest_path = Path(ols.artifact_manifest_json)
with open(manifest_path) as f:
    bundle_manifest = json.load(f)

all_input_metadata_csv = Path(bundle_manifest["files"]["all_input_metadata_csv"])
selected_input_metadata_csv = Path(bundle_manifest["files"]["selected_input_metadata_csv"])
output_metadata_csv = Path(bundle_manifest["files"]["output_metadata_csv"])
coef_std_csv = Path(bundle_manifest["files"]["coef_matrix_standardized_csv"])
coef_raw_csv = Path(bundle_manifest["files"]["coef_matrix_raw_scale_csv"])
x_scaler_csv = Path(bundle_manifest["files"]["x_standardization_csv"])
y_scaler_csv = Path(bundle_manifest["files"]["y_standardization_csv"])

all_input_metadata_df = pd.read_csv(all_input_metadata_csv)
selected_input_metadata_df = pd.read_csv(selected_input_metadata_csv)
output_metadata_df = pd.read_csv(output_metadata_csv)
coef_std_df = pd.read_csv(coef_std_csv, index_col=0)
coef_raw_df = pd.read_csv(coef_raw_csv, index_col=0)
x_transform_df = pd.read_csv(x_scaler_csv)
y_transform_df = pd.read_csv(y_scaler_csv)

all_input_names = all_input_metadata_df["input_name"].tolist()
selected_input_names = selected_input_metadata_df["input_name"].tolist()
retained_features = x_transform_df["feature_name"].tolist()
output_names = output_metadata_df["output_name"].tolist()

print("postfit_dir:", postfit_dir)
print("all_input_metadata_df shape:", all_input_metadata_df.shape)
print("selected_input_metadata_df shape:", selected_input_metadata_df.shape)
print("output_metadata_df shape:", output_metadata_df.shape)
print("coef_std_df shape:", coef_std_df.shape)
print("coef_raw_df shape:", coef_raw_df.shape)
print("x_transform_df shape:", x_transform_df.shape)
print("y_transform_df shape:", y_transform_df.shape)
print("all input features:", len(all_input_names))
print("selected input features:", len(selected_input_names))
print("retained features:", len(retained_features))
print("outputs:", len(output_names))


In [ ]:

def _save_compressed_parquet(df: pd.DataFrame, path: Path, compression: str = "zstd") -> Path:
    try:
        df.to_parquet(path, index=False, compression=compression)
    except Exception:
        fallback = path.with_suffix(".snappy.parquet")
        df.to_parquet(fallback, index=False, compression="snappy")
        return fallback
    return path

expected_coef_shape = (1 + len(retained_features), len(output_names))
assert coef_std_df.shape == expected_coef_shape, f"Standardized coefficient matrix shape mismatch: {coef_std_df.shape} vs {expected_coef_shape}"
assert coef_raw_df.shape == expected_coef_shape, f"Raw-scale coefficient matrix shape mismatch: {coef_raw_df.shape} vs {expected_coef_shape}"

assert all_input_metadata_df.shape[0] == X.shape[1], "Full input metadata length does not match the full modeled input count."
assert selected_input_metadata_df.shape[0] == len(fl.selected_features), "Selected input metadata length does not match the LASSO-selected input count."
assert x_transform_df.shape[0] == len(retained_features), "X transform vector length does not match retained feature count."
assert output_metadata_df.shape[0] == Y.shape[1], "Output metadata length does not match the full modeled output count."
assert y_transform_df.shape[0] == len(output_names), "Y transform vector length does not match output count."

assert all_input_names == list(X.columns), "Full input metadata names/order are not aligned to X.columns."
assert selected_input_names == list(fl.selected_features), "Selected input metadata names/order are not aligned to fl.selected_features."
assert output_names == list(Y.columns), "Output metadata names/order are not aligned to Y.columns."
assert y_transform_df["output_name"].tolist() == output_names, "Y transform vector order is not aligned to output metadata."
assert x_transform_df["feature_name"].tolist() == retained_features, "X transform vector order is not internally aligned."

expected_coef_rows = ["const"] + retained_features
assert coef_std_df.index.tolist() == expected_coef_rows, "Standardized coefficient row names are not aligned to retained features."
assert coef_raw_df.index.tolist() == expected_coef_rows, "Raw coefficient row names are not aligned to retained features."
assert coef_std_df.columns.tolist() == output_names, "Standardized coefficient columns are not aligned to output_names."
assert coef_raw_df.columns.tolist() == output_names, "Raw coefficient columns are not aligned to output_names."

assert np.isfinite(coef_std_df.to_numpy(dtype=np.float64)).all(), "Standardized coefficient matrix contains non-finite values."
assert np.isfinite(coef_raw_df.to_numpy(dtype=np.float64)).all(), "Raw coefficient matrix contains non-finite values."
assert np.isfinite(x_transform_df[["mean", "std"]].to_numpy(dtype=np.float64)).all(), "X transform vectors contain non-finite values."
assert np.isfinite(y_transform_df[["mean", "std"]].to_numpy(dtype=np.float64)).all(), "Y transform vectors contain non-finite values."

x_mean = x_transform_df["mean"].to_numpy(dtype=np.float64)
x_std = x_transform_df["std"].to_numpy(dtype=np.float64)
y_mean = y_transform_df["mean"].to_numpy(dtype=np.float64)
y_std = y_transform_df["std"].to_numpy(dtype=np.float64)

safe_x_std = np.where(np.abs(x_std) < 1e-15, 1.0, x_std)

beta_std = coef_std_df.loc[retained_features, output_names].to_numpy(dtype=np.float64)
intercept_std = coef_std_df.loc["const", output_names].to_numpy(dtype=np.float64)
beta_raw = coef_raw_df.loc[retained_features, output_names].to_numpy(dtype=np.float64)
intercept_raw = coef_raw_df.loc["const", output_names].to_numpy(dtype=np.float64)

beta_raw_from_std = (y_std[None, :] * beta_std) / safe_x_std[:, None]
intercept_raw_from_std = y_mean + y_std * intercept_std - (x_mean[:, None] * beta_raw_from_std).sum(axis=0)

np.testing.assert_allclose(beta_raw, beta_raw_from_std, rtol=1e-8, atol=1e-6)
np.testing.assert_allclose(intercept_raw, intercept_raw_from_std, rtol=1e-8, atol=1e-6)

shape_validation_df = pd.DataFrame(
    [
        {"artifact": "all_input_metadata", "rows": all_input_metadata_df.shape[0], "cols": all_input_metadata_df.shape[1]},
        {"artifact": "selected_input_metadata", "rows": selected_input_metadata_df.shape[0], "cols": selected_input_metadata_df.shape[1]},
        {"artifact": "output_metadata", "rows": output_metadata_df.shape[0], "cols": output_metadata_df.shape[1]},
        {"artifact": "coef_matrix_standardized", "rows": coef_std_df.shape[0], "cols": coef_std_df.shape[1]},
        {"artifact": "coef_matrix_raw_scale", "rows": coef_raw_df.shape[0], "cols": coef_raw_df.shape[1]},
        {"artifact": "x_transform_vectors", "rows": x_transform_df.shape[0], "cols": x_transform_df.shape[1]},
        {"artifact": "y_transform_vectors", "rows": y_transform_df.shape[0], "cols": y_transform_df.shape[1]},
        {"artifact": "ols.B", "rows": ols.B.shape[0], "cols": ols.B.shape[1]},
        {"artifact": "ols.se", "rows": ols.se.shape[0], "cols": ols.se.shape[1]},
        {"artifact": "ols.p", "rows": ols.p.shape[0], "cols": ols.p.shape[1]},
    ]
)

validation_report = {
    "full_input_count_expected": int(X.shape[1]),
    "full_input_count_metadata": int(all_input_metadata_df.shape[0]),
    "selected_input_count_expected": int(len(fl.selected_features)),
    "selected_input_count_metadata": int(selected_input_metadata_df.shape[0]),
    "retained_input_count": int(len(retained_features)),
    "output_count_expected": int(Y.shape[1]),
    "output_count_metadata": int(output_metadata_df.shape[0]),
    "coef_shape_expected": expected_coef_shape,
    "coef_shape_standardized": tuple(coef_std_df.shape),
    "coef_shape_raw": tuple(coef_raw_df.shape),
    "x_transform_length": int(x_transform_df.shape[0]),
    "y_transform_length": int(y_transform_df.shape[0]),
}
shape_validation_df


In [ ]:

# Save coefficient matrices, transform vectors, and metadata as compressed parquet files.
all_input_metadata_parquet = _save_compressed_parquet(
    all_input_metadata_df.copy(),
    postfit_dir / "all_input_metadata.parquet",
)
selected_input_metadata_parquet = _save_compressed_parquet(
    selected_input_metadata_df.copy(),
    postfit_dir / "selected_input_metadata.parquet",
)
output_metadata_parquet = _save_compressed_parquet(
    output_metadata_df.copy(),
    postfit_dir / "output_metadata.parquet",
)
coef_std_parquet = _save_compressed_parquet(
    coef_std_df.reset_index().rename(columns={"coefficient_row_name": "coef_row_name"}),
    postfit_dir / "coef_matrix_standardized.parquet",
)
coef_raw_parquet = _save_compressed_parquet(
    coef_raw_df.reset_index().rename(columns={"coefficient_row_name": "coef_row_name"}),
    postfit_dir / "coef_matrix_raw_scale.parquet",
)
x_transform_parquet = _save_compressed_parquet(
    x_transform_df.copy(),
    postfit_dir / "x_standardization.parquet",
)
y_transform_parquet = _save_compressed_parquet(
    y_transform_df.copy(),
    postfit_dir / "y_standardization.parquet",
)
shape_validation_parquet = _save_compressed_parquet(
    shape_validation_df.copy(),
    postfit_dir / "artifact_shape_validation.parquet",
)

if "selected_input_structure_df" in globals():
    selected_input_structure_parquet = _save_compressed_parquet(
        selected_input_structure_df.copy(),
        postfit_dir / "selected_input_structure.parquet",
    )
    selected_input_type_summary_parquet = _save_compressed_parquet(
        selected_input_type_summary_df.copy(),
        postfit_dir / "selected_input_type_summary.parquet",
    )
else:
    selected_input_structure_parquet = None
    selected_input_type_summary_parquet = None

with open(postfit_dir / "artifact_validation_report.json", "w") as f:
    json.dump(validation_report, f, indent=2)

print("Saved parquet artifacts:")
print(" -", all_input_metadata_parquet)
print(" -", selected_input_metadata_parquet)
print(" -", output_metadata_parquet)
print(" -", coef_std_parquet)
print(" -", coef_raw_parquet)
print(" -", x_transform_parquet)
print(" -", y_transform_parquet)
print(" -", shape_validation_parquet)
if selected_input_structure_parquet is not None:
    print(" -", selected_input_structure_parquet)
    print(" -", selected_input_type_summary_parquet)
print(" -", postfit_dir / "artifact_validation_report.json")


In [ ]:

# Collect screening/final nRMSE summaries with bootstrap CIs from the pipeline outputs.
final_valid_dir = Path(ols.validation_dir)
screen_valid_dir = Path(fl.validation_dir)

nrmse_summary_frames = []

for candidate in [
    screen_valid_dir / "lasso_internal_nrmse_summary.csv",
    screen_valid_dir / "lasso_holdout_nrmse_summary.csv",
    final_valid_dir / "nrmse_summary.csv",
]:
    if candidate.exists():
        df_tmp = pd.read_csv(candidate)
        df_tmp["source_file"] = str(candidate)
        nrmse_summary_frames.append(df_tmp)

if nrmse_summary_frames:
    nrmse_summary_all_df = pd.concat(nrmse_summary_frames, ignore_index=True)
else:
    nrmse_summary_all_df = pd.DataFrame(
        columns=[
            "stage",
            "evaluation_set",
            "model",
            "nrmse_macro",
            "nrmse_ci_lower",
            "nrmse_ci_upper",
            "bootstrap_mean",
            "bootstrap_std",
            "n_boot",
            "alpha",
            "k_used_outputs",
            "k_total_outputs",
            "rmse_macro",
            "n_eval_rows",
            "source_file",
        ]
    )

nrmse_summary_csv = postfit_dir / "nrmse_summary_all_models.csv"
nrmse_summary_parquet = _save_compressed_parquet(
    nrmse_summary_all_df.copy(),
    postfit_dir / "nrmse_summary_all_models.parquet",
)
nrmse_summary_all_df.to_csv(nrmse_summary_csv, index=False)

print("Saved combined nRMSE summary files:")
print(" -", nrmse_summary_csv)
print(" -", nrmse_summary_parquet)
nrmse_summary_all_df


In [ ]:

# Reconstruct holdout predictions from the exported raw-scale coefficient matrix and validate
# against the standardized prediction formula.
X_eval = X_ho.loc[:, retained_features].copy()
Y_true_eval = Y_ho.loc[:, output_names].copy()

X_eval_matrix = X_eval.to_numpy(dtype=np.float64)
Yhat_raw_direct = X_eval_matrix @ beta_raw + intercept_raw[None, :]

X_eval_std = (X_eval_matrix - x_mean[None, :]) / safe_x_std[None, :]
Yhat_std = X_eval_std @ beta_std + intercept_std[None, :]
Yhat_raw_from_std = y_mean[None, :] + y_std[None, :] * Yhat_std

abs_diff = np.abs(Yhat_raw_direct - Yhat_raw_from_std)
rel_diff = abs_diff / np.maximum(np.abs(Yhat_raw_from_std), 1e-12)

print("max abs diff:", float(abs_diff.max()))
print("max rel diff:", float(rel_diff.max()))
print("mean abs diff:", float(abs_diff.mean()))

np.testing.assert_allclose(Yhat_raw_direct, Yhat_raw_from_std, rtol=1e-8, atol=1e-6)

Y_pred_eval = pd.DataFrame(Yhat_raw_direct, index=Y_true_eval.index, columns=output_names)

eval_alignment_df = pd.DataFrame(
    {
        "n_rows": [len(Y_true_eval)],
        "n_outputs": [Y_true_eval.shape[1]],
        "index_match": [bool(Y_true_eval.index.equals(Y_pred_eval.index))],
        "column_match": [bool(Y_true_eval.columns.equals(Y_pred_eval.columns))],
    }
)
eval_alignment_df


In [ ]:

# 2050 summary for each variable in `variables`, using the external holdout set.
summary_rows = []

for variable in variables:
    col_2050 = f"{variable}_2050"
    if col_2050 not in Y_true_eval.columns:
        continue

    known = Y_true_eval[col_2050].astype(np.float64)
    pred = Y_pred_eval[col_2050].astype(np.float64)

    sst = float(np.sum((known - known.mean()) ** 2))
    sse = float(np.sum((known - pred) ** 2))
    r2 = np.nan if sst <= 1e-15 else 1.0 - (sse / sst)

    summary_rows.append(
        {
            "variable": variable,
            "output_col_2050": col_2050,
            "n_holdout": int(len(known)),
            "mean_known": float(known.mean()),
            "mean_predicted": float(pred.mean()),
            "median_known": float(known.median()),
            "median_predicted": float(pred.median()),
            "min_known": float(known.min()),
            "min_predicted": float(pred.min()),
            "max_known": float(known.max()),
            "max_predicted": float(pred.max()),
            "mae": float(np.mean(np.abs(known - pred))),
            "rmse": float(np.sqrt(np.mean((known - pred) ** 2))),
            "r2_holdout_2050": float(r2) if np.isfinite(r2) else np.nan,
        }
    )

summary_2050_df = (
    pd.DataFrame(summary_rows)
    .sort_values("variable")
    .reset_index(drop=True)
)

summary_2050_df = summary_2050_df[
    [
        "variable",
        "output_col_2050",
        "n_holdout",
        "mean_known",
        "mean_predicted",
        "median_known",
        "median_predicted",
        "min_known",
        "min_predicted",
        "max_known",
        "max_predicted",
        "mae",
        "rmse",
        "r2_holdout_2050",
    ]
]

summary_2050_df


In [ ]:

summary_2050_csv = postfit_dir / "summary_2050_holdout.csv"
summary_2050_parquet = _save_compressed_parquet(summary_2050_df.copy(), postfit_dir / "summary_2050_holdout.parquet")

summary_2050_df.to_csv(summary_2050_csv, index=False)

print("Saved 2050 summary files:")
print(" -", summary_2050_csv)
print(" -", summary_2050_parquet)


In [ ]:

# Trajectory overlays for each variable across all available years.
def _slugify(text: str) -> str:
    text = re.sub(r"[^A-Za-z0-9]+", "_", text).strip("_")
    return text.lower()

sample_n = min(100, len(Y_true_eval))
rs = np.random.RandomState(123)
sample_positions = np.sort(rs.choice(np.arange(len(Y_true_eval)), size=sample_n, replace=False))
sample_index = Y_true_eval.index[sample_positions]

trajectory_index_records = []

for variable in variables:
    var_cols = [f"{variable}_{year}" for year in years if f"{variable}_{year}" in Y_true_eval.columns]
    if len(var_cols) == 0:
        continue

    year_axis = [int(col.rsplit("_", 1)[1]) for col in var_cols]
    Y_true_mat = Y_true_eval.loc[sample_index, var_cols].to_numpy(dtype=np.float64)
    Y_pred_mat = Y_pred_eval.loc[sample_index, var_cols].to_numpy(dtype=np.float64)

    fig, ax = plt.subplots(figsize=(10, 5))
    for row in Y_true_mat:
        ax.plot(year_axis, row, color="black", alpha=0.15, linewidth=1.0, zorder=1)
    for row in Y_pred_mat:
        ax.plot(year_axis, row, color="red", alpha=0.22, linewidth=1.0, zorder=2)

    ax.set_title(variable)
    ax.set_xlabel("Year")
    ax.set_ylabel("Output value")
    ax.grid(True, alpha=0.25)

    png_path = postfit_dir / f"{_slugify(variable)}_trajectory_overlay.png"
    fig.tight_layout()
    fig.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.show()

    trajectory_index_records.append(
        {
            "variable": variable,
            "n_sampled_trajectories": int(sample_n),
            "n_years": int(len(var_cols)),
            "plot_path": str(png_path),
        }
    )

trajectory_index_df = pd.DataFrame(trajectory_index_records)
trajectory_index_df


In [ ]:

trajectory_index_csv = postfit_dir / "trajectory_plot_index.csv"
trajectory_index_parquet = _save_compressed_parquet(
    trajectory_index_df.copy(),
    postfit_dir / "trajectory_plot_index.parquet",
)
trajectory_index_df.to_csv(trajectory_index_csv, index=False)

print("Saved trajectory index files:")
print(" -", trajectory_index_csv)
print(" -", trajectory_index_parquet)
